# LibreYOLOXs Model Analysis for Qualcomm® Snapdragon Devices

This notebook is the final benchmarking and validation workflow on deploying optimized LibreYOLOXs models to Qualcomm® Snapdragon Devices.

## Objective

The goal is to compare the generated LibreYOLOXs artifacts across:

- **QAI Hub FP32 / INT8 DLC**
- **QAIRT FP32 / INT8 DLC**
- **SNPE FP32 / INT8 DLC**

We analyze:

- model size
- latency
- throughput
- backend/runtime behavior
- quantization impact
- output similarity
- detection accuracy

## Why this analysis matters

Quantization and backend selection are central Edge AI decisions. A model that is smaller and faster is not automatically the best deployment candidate if its outputs drift too much or its detection accuracy drops materially. This notebook therefore combines **systems benchmarking** and **model validation** in one reproducible workflow.

## Hardware and backend relevance

The target device is the **Dragonwing RB3 Gen 2 Vision Kit**, accessed through **QAI Hub** when cloud execution is available. We compare:

- **CPU** for portability and baseline behavior
- **GPU** for floating-point acceleration
- **HTP/DSP** for Qualcomm integer-optimized edge inference

These comparisons are especially important for edge deployment because the best backend can vary depending on precision, runtime support, and accuracy-performance trade-offs.

## Environment Setup

The notebook uses only relative paths and creates all report artifacts locally under `reports/`.

Optional dependencies are handled gracefully:

- `qai_hub` for cloud profiling and inference
- `pycocotools` for COCO mAP evaluation
- QAIRT / SNPE CLIs for local DLC inspection

If any optional dependency is missing, the notebook continues and records placeholders in the result tables rather than failing early.

In [1]:
import json
import math
import os
import re
import shutil
import subprocess

# Force a non-interactive backend for headless Docker execution.
import matplotlib
matplotlib.use("Agg", force=True)

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Any

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):  # type: ignore[override]
        return iterable

try:
    import qai_hub as hub
except Exception as exc:
    hub = None
    QAI_HUB_IMPORT_ERROR = exc
else:
    QAI_HUB_IMPORT_ERROR = None

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    PYCOCOTOOLS_AVAILABLE = True
except Exception as exc:
    COCO = None
    COCOeval = None
    PYCOCOTOOLS_AVAILABLE = False
    PYCOCOTOOLS_IMPORT_ERROR = exc
else:
    PYCOCOTOOLS_IMPORT_ERROR = None

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

In [2]:
MODEL_ROOT = Path("../models")
QAIHUB_FP32_DLC = MODEL_ROOT / "qaihub/LibreYOLOXs_fp32.dlc"
QAIHUB_INT8_DLC = MODEL_ROOT / "qaihub/LibreYOLOXs_int8.dlc"

QAIRT_FP32_DLC = MODEL_ROOT / "qairt/LibreYOLOXs_fp32.dlc"
QAIRT_INT8_DLC = MODEL_ROOT / "qairt/LibreYOLOXs_int8.dlc"

SNPE_FP32_DLC = MODEL_ROOT / "snpe/LibreYOLOXs_fp32.dlc"
SNPE_INT8_DLC = MODEL_ROOT / "snpe/LibreYOLOXs_int8.dlc"

DEVICE = hub.Device("Dragonwing RB3 Gen 2 Vision Kit") if hub is not None else None

DATASET_ROOT = Path("dataset")
COCO_IMAGES_DIR = DATASET_ROOT / "val2017"
COCO_ANNOTATIONS_DIR = DATASET_ROOT / "annotations"
COCO_ANNOTATION_FILE = COCO_ANNOTATIONS_DIR / "instances_val2017.json"

SUBSET_ROOT = Path("subset")
CALIB_DIR = SUBSET_ROOT / "calib"
VAL_DIR = SUBSET_ROOT / "val"

REPORTS_ROOT = Path("reports")
FIGURES_DIR = REPORTS_ROOT / "figures"
TABLES_DIR = REPORTS_ROOT / "tables"
DLC_INFO_DIR = REPORTS_ROOT / "dlc_info"
INFERENCE_OUTPUTS_DIR = REPORTS_ROOT / "inference_outputs"
PROFILE_ARTIFACTS_DIR = REPORTS_ROOT / "profiles"

for directory in [
    SUBSET_ROOT,
    CALIB_DIR,
    VAL_DIR,
    REPORTS_ROOT,
    FIGURES_DIR,
    TABLES_DIR,
    DLC_INFO_DIR,
    INFERENCE_OUTPUTS_DIR,
    PROFILE_ARTIFACTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

OPTIONAL_DEPENDENCIES_DF = pd.DataFrame(
    [
        {
            "Dependency": "qai_hub",
            "Available": hub is not None,
            "Notes": None if hub is not None else str(QAI_HUB_IMPORT_ERROR),
        },
        {
            "Dependency": "pycocotools",
            "Available": PYCOCOTOOLS_AVAILABLE,
            "Notes": None if PYCOCOTOOLS_AVAILABLE else str(PYCOCOTOOLS_IMPORT_ERROR),
        },
        {
            "Dependency": "qairt-dlc-info CLI",
            "Available": shutil.which("qairt-dlc-info") is not None,
            "Notes": None,
        },
        {
            "Dependency": "snpe-dlc-info CLI",
            "Available": shutil.which("snpe-dlc-info") is not None,
            "Notes": None,
        },
    ]
)

OPTIONAL_DEPENDENCIES_DF

,Dependency,Available,Notes
0,qai_hub,True,None
1,pycocotools,True,None
2,qairt-dlc-info CLI,True,None
3,snpe-dlc-info CLI,True,None


## Model Inventory

We first build a registry covering all expected model artifacts. This gives us a single source of truth for:

- model name
- format
- precision
- source pipeline
- filesystem path
- file size in MB

Missing files are retained in the table so that the notebook can still document incomplete environments clearly.

In [3]:
def get_file_size_mb(path: Path) -> float:
    """
    Return the size of a file in megabytes (MB).

    Args:
        path (Path): Path to the target file.

    Returns:
        float: File size in megabytes. Returns NaN if the file does not exist.
    """
    if not path.exists():
        return float("nan")

    return path.stat().st_size / (1024 ** 2)


MODEL_SPECS = [
    {
        "Model": "LibreYOLOXs QAI Hub FP32 DLC",
        "Format": "DLC",
        "Precision": "FP32",
        "Pipeline": "QAI Hub",
        "Path": QAIHUB_FP32_DLC,
    },
    {
        "Model": "LibreYOLOXs QAI Hub INT8 DLC",
        "Format": "DLC",
        "Precision": "INT8",
        "Pipeline": "QAI Hub",
        "Path": QAIHUB_INT8_DLC,
    },
    {
        "Model": "LibreYOLOXs QAIRT FP32 DLC",
        "Format": "DLC",
        "Precision": "FP32",
        "Pipeline": "QAIRT",
        "Path": QAIRT_FP32_DLC,
    },
    {
        "Model": "LibreYOLOXs QAIRT INT8 DLC",
        "Format": "DLC",
        "Precision": "INT8",
        "Pipeline": "QAIRT",
        "Path": QAIRT_INT8_DLC,
    },
    {
        "Model": "LibreYOLOXs SNPE FP32 DLC",
        "Format": "DLC",
        "Precision": "FP32",
        "Pipeline": "SNPE",
        "Path": SNPE_FP32_DLC,
    },
    {
        "Model": "LibreYOLOXs SNPE INT8 DLC",
        "Format": "DLC",
        "Precision": "INT8",
        "Pipeline": "SNPE",
        "Path": SNPE_INT8_DLC,
    },
]

model_inventory_df = pd.DataFrame(MODEL_SPECS)
model_inventory_df["Path"] = model_inventory_df["Path"].map(lambda path: str(path))
model_inventory_df["Exists"] = model_inventory_df["Path"].map(lambda path: Path(path).exists())
model_inventory_df["File_Size_MB"] = model_inventory_df["Path"].map(lambda path: get_file_size_mb(Path(path)))

model_inventory_df

,Model,Format,Precision,Pipeline,Path,Exists,File_Size_MB
0,LibreYOLOXs QAI Hub FP32 DLC,DLC,FP32,QAI Hub,../models/qaihub/LibreYOLOXs_fp32.dlc,True,34.509594
1,LibreYOLOXs QAI Hub INT8 DLC,DLC,INT8,QAI Hub,../models/qaihub/LibreYOLOXs_int8.dlc,True,9.518322
2,LibreYOLOXs QAIRT FP32 DLC,DLC,FP32,QAIRT,../models/qairt/LibreYOLOXs_fp32.dlc,True,34.508976
3,LibreYOLOXs QAIRT INT8 DLC,DLC,INT8,QAIRT,../models/qairt/LibreYOLOXs_int8.dlc,True,8.841442
4,LibreYOLOXs SNPE FP32 DLC,DLC,FP32,SNPE,../models/snpe/LibreYOLOXs_fp32.dlc,True,34.505108
5,LibreYOLOXs SNPE INT8 DLC,DLC,INT8,SNPE,../models/snpe/LibreYOLOXs_int8.dlc,True,8.843380


## Input Dataset Preparation

LibreYOLOXs preprocessing in this project uses the following contract:

- **BGR input**
- **float32 values in the 0-255 range**
- **CHW layout**
- **top-left letterbox**
- **no `/255` normalization**
- **no BGR→RGB conversion**

The next cells prepare:

- `subset/calib/`
- `subset/val/`
- calibration and validation `.raw` tensors
- `metadata.csv`
- `filenames.txt`

If COCO `val2017` is not present locally, the notebook can either print placement instructions or optionally download it.

In [4]:
def letterbox(
    image: np.ndarray,
    target: int = 640,
    pad_value: int = 114,
) -> tuple[np.ndarray, float]:
    """
    Resize an image while preserving its aspect ratio and apply top-left padding
    to fit a square target size.

    The resized image is placed in the top-left corner of the padded output image.

    Args:
        image (np.ndarray): Input image in HWC format.
        target (int, optional): Target square size for the output image.
            Defaults to 640.
        pad_value (int, optional): Padding pixel value used to fill empty areas.
            Defaults to 114.

    Returns:
        tuple[np.ndarray, float]:
            - The padded square image as a float32 NumPy array.
            - The resize ratio applied to the original image.

    Raises:
        ValueError: If the input image is None.
    """
    if image is None:
        raise ValueError("Received an empty image.")

    h, w = image.shape[:2]
    ratio = min(target / h, target / w)
    new_w, new_h = int(round(w * ratio)), int(round(h * ratio))

    resized = cv.resize(image, (new_w, new_h), interpolation=cv.INTER_LINEAR)

    padded = np.full((target, target, 3), pad_value, dtype=np.float32)
    padded[:new_h, :new_w] = resized.astype(np.float32)

    return padded, ratio


def preprocess(
    original_image: np.ndarray,
    size: int = 640,
) -> tuple[np.ndarray, float]:
    """
    Preprocess an image for YOLOX inference.

    The preprocessing pipeline performs:
    - Aspect-ratio-preserving letterbox resize
    - Top-left padding to a square image
    - HWC to CHW layout conversion
    - Conversion to float32 format
    - Preservation of BGR color ordering
    - Preservation of the 0-255 pixel value range

    Args:
        original_image (np.ndarray): Input image in BGR format using HWC layout.
        size (int, optional): Target square image size. Defaults to 640.

    Returns:
        tuple[np.ndarray, float]:
            - Preprocessed image tensor in CHW float32 format.
            - Resize ratio applied during letterboxing.
    """
    padded, ratio = letterbox(original_image, target=size)

    chw = padded.transpose(2, 0, 1).astype(np.float32, copy=False)

    return chw, ratio


def run_subprocess(command: list[str]) -> subprocess.CompletedProcess:
    """
    Execute a subprocess command and return the completed process result.

    The command is executed without raising exceptions on non-zero exit codes.
    Standard output and standard error are captured as text.

    Args:
        command (list[str]): Command and arguments to execute.

    Returns:
        subprocess.CompletedProcess: Result object containing the executed
        command, return code, standard output, and standard error.
    """
    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )


def ensure_coco_val2017(auto_download: bool = False) -> dict[str, Any]:
    """
    Ensure that the COCO val2017 dataset and annotations are available locally.

    This function verifies whether the COCO val2017 images and annotation files
    exist in the expected directories. If they are missing and automatic download
    is enabled, the dataset archives are downloaded and extracted.

    Args:
        auto_download (bool, optional): Whether to automatically download and
            extract the dataset if it is missing. Defaults to False.

    Returns:
        dict[str, Any]: Dictionary containing:
            - "ready" (bool): Indicates whether the dataset is available.
            - "notes" (str): Informational or error message describing the result.
    """
    image_ready = COCO_IMAGES_DIR.exists()
    ann_ready = COCO_ANNOTATION_FILE.exists()

    if image_ready and ann_ready:
        return {
            "ready": True,
            "notes": "Using existing COCO val2017 images and annotations.",
        }

    if not auto_download:
        notes = (
            "COCO val2017 data was not found. Place images under "
            f"{COCO_IMAGES_DIR} and annotations at {COCO_ANNOTATION_FILE}, "
            "or set AUTO_DOWNLOAD_COCO = True in the next cell."
        )

        return {
            "ready": False,
            "notes": notes,
        }

    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    downloads = [
        (
            "http://images.cocodataset.org/zips/val2017.zip",
            DATASET_ROOT / "val2017.zip",
        ),
        (
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            DATASET_ROOT / "annotations_trainval2017.zip",
        ),
    ]

    for url, archive_path in downloads:
        if archive_path.exists():
            continue

        result = run_subprocess(
            ["curl", "-L", "-o", str(archive_path), url]
        )

        if result.returncode != 0:
            return {
                "ready": False,
                "notes": (
                    f"Failed to download {url}: "
                    f"{result.stderr.strip()}"
                ),
            }

    for archive_path in [
        DATASET_ROOT / "val2017.zip",
        DATASET_ROOT / "annotations_trainval2017.zip",
    ]:
        result = run_subprocess(
            ["unzip", "-o", str(archive_path), "-d", str(DATASET_ROOT)]
        )

        if result.returncode != 0:
            return {
                "ready": False,
                "notes": (
                    f"Failed to unzip {archive_path.name}: "
                    f"{result.stderr.strip()}"
                ),
            }

    return {
        "ready": (
            COCO_IMAGES_DIR.exists()
            and COCO_ANNOTATION_FILE.exists()
        ),
        "notes": "COCO val2017 download attempted.",
    }


def extract_image_id(image_path: Path) -> int:
    """
    Extract the numeric image ID from an image filename.

    The function removes all non-digit characters from the file stem and
    converts the remaining digits into an integer.

    Args:
        image_path (Path): Path to the image file.

    Returns:
        int: Extracted numeric image ID.

    Raises:
        ValueError: If the filename does not contain any numeric characters.
    """
    digits = re.sub(r"\D", "", image_path.stem)

    if not digits:
        raise ValueError(
            f"Could not infer image_id from {image_path.name}"
        )

    return int(digits)


def write_raw_manifest(
    raw_paths: list[Path],
    manifest_path: Path,
) -> None:
    """
    Write a manifest file containing one file path per line.

    The manifest is typically used to store references to raw input files
    for later processing. A trailing newline is added if the list is not empty.

    Args:
        raw_paths (list[Path]): List of file paths to include in the manifest.
        manifest_path (Path): Output path for the manifest file.

    Returns:
        None
    """
    manifest_path.write_text(
        "\n".join(str(path) for path in raw_paths)
        + ("\n" if raw_paths else "")
    )


def prepare_subset(
    calib_count: int = 1000,
    val_count: int = 100,
    seed: int = 42,
    auto_download: bool = False,
    overwrite: bool = False,
) -> dict[str, Any]:
    """
    Prepare calibration and validation subsets from the COCO val2017 dataset.

    This function checks whether the COCO val2017 dataset is available, optionally
    downloads it, samples calibration and validation images, preprocesses them into
    RAW tensor files, and writes manifest files for later inference or profiling.

    It also creates a validation metadata CSV file containing the original image
    path, RAW file path, image ID, original dimensions, preprocessing ratio, and
    input format description.

    Args:
        calib_count (int, optional): Number of images to use for the calibration
            subset. Defaults to 1000.
        val_count (int, optional): Number of images to use for the validation
            subset. Defaults to 100.
        seed (int, optional): Random seed used to shuffle and sample images.
            Defaults to 42.
        auto_download (bool, optional): Whether to automatically download COCO
            val2017 if it is not found locally. Defaults to False.
        overwrite (bool, optional): Whether to overwrite existing RAW files.
            Defaults to False.

    Returns:
        dict[str, Any]: Dictionary containing:
            - "ready" (bool): Whether the subset preparation completed.
            - "notes" (str): Status message from the dataset readiness check.
            - "calib_raw_paths" (list[Path]): Paths to calibration RAW files.
            - "val_raw_paths" (list[Path]): Paths to validation RAW files.
            - "metadata_df" (pd.DataFrame): Validation metadata table.

            If the dataset is not ready, only "ready" and "notes" are returned.

    Raises:
        ValueError: If there are not enough COCO images to build the requested
            calibration and validation subsets.
    """
    readiness = ensure_coco_val2017(auto_download=auto_download)

    if not readiness["ready"]:
        print(readiness["notes"])
        return {
            "ready": False,
            "notes": readiness["notes"],
        }

    image_paths = sorted(COCO_IMAGES_DIR.glob("*.jpg"))

    if len(image_paths) < calib_count + val_count:
        raise ValueError(
            f"Need at least {calib_count + val_count} COCO images, "
            f"found {len(image_paths)}."
        )

    rng = np.random.default_rng(seed)
    shuffled = image_paths.copy()
    rng.shuffle(shuffled)

    calib_paths = shuffled[:calib_count]
    val_paths = shuffled[calib_count:calib_count + val_count]

    if overwrite:
        for folder in [CALIB_DIR, VAL_DIR]:
            for raw_file in folder.glob("*.raw"):
                raw_file.unlink()

    calib_raw_paths: list[Path] = []

    for image_path in tqdm(
        calib_paths,
        desc="Preparing calibration RAW files",
    ):
        raw_path = CALIB_DIR / f"{image_path.stem}.raw"

        if overwrite or not raw_path.exists():
            image = cv.imread(str(image_path))
            tensor, _ = preprocess(image)
            tensor.reshape(1, 3, 640, 640).tofile(raw_path)

        calib_raw_paths.append(raw_path)

    metadata_rows: list[dict[str, Any]] = []
    val_raw_paths: list[Path] = []

    for image_path in tqdm(
        val_paths,
        desc="Preparing validation RAW files",
    ):
        raw_path = VAL_DIR / f"{image_path.stem}.raw"
        image = cv.imread(str(image_path))

        if image is None:
            continue

        tensor, ratio = preprocess(image)

        if overwrite or not raw_path.exists():
            tensor.reshape(1, 3, 640, 640).tofile(raw_path)

        metadata_rows.append(
            {
                "raw_path": str(raw_path),
                "image_path": str(image_path),
                "image_id": extract_image_id(image_path),
                "orig_w": int(image.shape[1]),
                "orig_h": int(image.shape[0]),
                "ratio": float(ratio),
                "input_format": "BGR float32 CHW 0-255 top-left letterbox",
            }
        )

        val_raw_paths.append(raw_path)

    metadata_df = pd.DataFrame(metadata_rows)
    metadata_df.to_csv(VAL_DIR / "metadata.csv", index=False)

    write_raw_manifest(calib_raw_paths, CALIB_DIR / "filenames.txt")
    write_raw_manifest(val_raw_paths, VAL_DIR / "filenames.txt")

    return {
        "ready": True,
        "notes": readiness["notes"],
        "calib_raw_paths": calib_raw_paths,
        "val_raw_paths": val_raw_paths,
        "metadata_df": metadata_df,
    }

In [5]:
AUTO_DOWNLOAD_COCO = False
CALIBRATION_SAMPLE_COUNT = 1000
VALIDATION_SAMPLE_COUNT = 100

subset_state = prepare_subset(
    calib_count=CALIBRATION_SAMPLE_COUNT,
    val_count=VALIDATION_SAMPLE_COUNT,
    seed=42,
    auto_download=AUTO_DOWNLOAD_COCO,
    overwrite=False,
)

if subset_state.get("ready"):
    print(subset_state["notes"])
    subset_state["metadata_df"].head()
else:
    print(subset_state["notes"])

Preparing calibration RAW files:   0%|          | 0/1000 [00:00<?, ?it/s]

Preparing validation RAW files:   0%|          | 0/100 [00:00<?, ?it/s]

Using existing COCO val2017 images and annotations.


## QAI Hub Profiling

This section uses **QAI Hub `submit_profile_job()`** to measure latency and throughput across available backends.

Planned coverage:

- **FP32 DLCs**: CPU, GPU
- **INT8 DLCs**: CPU, HTP/DSP

Because exact backend flags can vary with SDK / service version, the helper functions below keep the runtime option handling explicit and easy to adapt.

In [6]:
def sanitize_name(name: str) -> str:
    """
    Sanitize a string for safe filesystem or identifier usage.

    All characters except letters, numbers, underscores, periods,
    and hyphens are replaced with underscores.

    Args:
        name (str): Input string to sanitize.

    Returns:
        str: Sanitized string containing only safe characters.
    """
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


def qaihub_backend_option(backend: str) -> str:
    """
    Convert a backend name into a Qualcomm AI Hub compute unit option.

    The function normalizes common backend aliases and returns the
    corresponding command-line argument used by Qualcomm AI Hub tools.

    Supported mappings:
    - "cpu" -> "--compute_unit cpu"
    - "gpu" -> "--compute_unit gpu"
    - "htp", "dsp", "htp/dsp", "htp_dsp" -> "--compute_unit npu"

    Any other backend name is passed through directly after normalization.

    Args:
        backend (str): Backend name or alias.

    Returns:
        str: Qualcomm AI Hub compute unit command-line option.
    """
    backend_key = backend.strip().lower()

    if backend_key == "cpu":
        return "--compute_unit cpu"

    if backend_key == "gpu":
        return "--compute_unit gpu"

    if backend_key in {"htp", "dsp", "htp/dsp", "htp_dsp"}:
        return "--compute_unit npu"

    return f"--compute_unit {backend_key}"


def load_qaihub_token() -> str | None:
    """
    Load the Qualcomm AI Hub API token from environment variables.

    If the `python-dotenv` package is available and a local `.env` file exists,
    the environment variables are loaded before retrieving the token.

    Returns:
        str | None:
            - The value of the `QAI_HUB_API_TOKEN` environment variable.
            - None if the token is not defined.
    """
    if load_dotenv is not None and Path(".env").exists():
        load_dotenv(".env", override=False)

    return os.getenv("QAI_HUB_API_TOKEN")


def configure_qaihub_cli() -> tuple[bool, str]:
    """
    Configure the Qualcomm AI Hub CLI using the API token from the environment.

    The function loads the `QAI_HUB_API_TOKEN`, verifies that the `qai-hub`
    command-line interface is installed, and executes the CLI configuration
    command.

    Returns:
        tuple[bool, str]:
            - A boolean indicating whether the configuration succeeded.
            - A status or error message describing the result.
    """
    token = load_qaihub_token()

    if not token:
        return (
            False,
            "QAI_HUB_API_TOKEN was not found in the environment or .env.",
        )

    if shutil.which("qai-hub") is None:
        return (
            False,
            "`qai-hub` CLI is not installed; skipping CLI configuration.",
        )

    result = subprocess.run(
        ["qai-hub", "configure", "--api_token", token],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        return (
            False,
            result.stderr.strip()
            or "Failed to configure qai-hub CLI.",
        )

    return True, "qai-hub CLI configured."


def get_qaihub_client() -> tuple[Any | None, list[str]]:
    """
    Create and return a Qualcomm AI Hub client instance.

    The function verifies whether the `qai_hub` package was imported
    successfully, attempts to configure the Qualcomm AI Hub CLI, and then
    creates a `hub.Client` instance.

    Returns:
        tuple[Any | None, list[str]]:
            - The Qualcomm AI Hub client instance, or None if client creation
              failed.
            - A list of status or error messages collected during setup.
    """
    notes: list[str] = []

    if hub is None:
        notes.append(f"qai_hub import failed: {QAI_HUB_IMPORT_ERROR}")
        return None, notes

    configured, message = configure_qaihub_cli()
    notes.append(message)

    try:
        client = hub.Client()
    except Exception as exc:
        notes.append(f"Failed to create hub.Client(): {exc}")
        return None, notes

    return client, notes


def to_serializable(obj: Any) -> Any:
    """
    Convert an object into a JSON-serializable representation.

    This function recursively converts common Python and NumPy objects into
    structures that can be safely serialized, such as dictionaries, lists,
    strings, and primitive values.

    Supported conversions include:
    - pathlib.Path -> str
    - numpy.ndarray -> metadata dictionary
    - list/tuple -> recursive list conversion
    - dict -> recursive dictionary conversion
    - custom objects with __dict__ -> recursive attribute conversion

    Args:
        obj (Any): Object to convert into a serializable representation.

    Returns:
        Any: JSON-serializable representation of the input object.
    """
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, np.ndarray):
        return {
            "type": "ndarray",
            "shape": list(obj.shape),
            "dtype": str(obj.dtype),
        }

    if isinstance(obj, (list, tuple)):
        return [to_serializable(item) for item in obj]

    if isinstance(obj, dict):
        return {
            str(key): to_serializable(value)
            for key, value in obj.items()
        }

    if hasattr(obj, "__dict__"):
        return to_serializable(vars(obj))

    return str(obj)


def save_json(path: Path, payload: Any) -> None:
    """
    Save a Python object as a formatted JSON file.

    The target directory is created automatically if it does not exist.
    The payload is recursively converted into a JSON-serializable structure
    before being written to disk.

    Args:
        path (Path): Output path for the JSON file.
        payload (Any): Python object to serialize and save.

    Returns:
        None
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:
        json.dump(
            to_serializable(payload),
            handle,
            indent=2,
        )


def deep_find_numbers(
    payload: Any,
    keywords: tuple[str, ...],
) -> list[tuple[str, float]]:
    """
    Recursively search nested data structures for numeric values whose keys
    contain specific keywords.

    The function traverses dictionaries and lists, collecting numeric values
    associated with dictionary keys that partially match any of the provided
    keywords. Each result includes the hierarchical path to the matched value.

    Args:
        payload (Any): Nested structure to search, typically composed of
            dictionaries and lists.
        keywords (tuple[str, ...]): Keywords used to match dictionary keys.
            Matching is case-insensitive and based on substring inclusion.

    Returns:
        list[tuple[str, float]]:
            List of tuples containing:
            - The hierarchical path to the matched value.
            - The numeric value converted to float.
    """
    hits: list[tuple[str, float]] = []

    def _walk(obj: Any, prefix: str = "") -> None:
        """
        Recursively traverse nested structures and collect matching values.

        Args:
            obj (Any): Current object being traversed.
            prefix (str, optional): Hierarchical key path prefix.
                Defaults to an empty string.

        Returns:
            None
        """
        if isinstance(obj, dict):
            for key, value in obj.items():
                lower_key = key.lower()
                child_prefix = (
                    f"{prefix}.{key}" if prefix else key
                )

                if (
                    any(keyword in lower_key for keyword in keywords)
                    and isinstance(value, (int, float))
                ):
                    hits.append((child_prefix, float(value)))

                _walk(value, child_prefix)

        elif isinstance(obj, list):
            for index, value in enumerate(obj):
                _walk(value, f"{prefix}[{index}]")

    _walk(payload)

    return hits


def parse_latency_ms(payload: Any) -> float | None:
    """
    Extract latency information from a nested payload and return it in
    milliseconds.

    The function searches recursively for numeric values associated with keys
    containing the word "latency". If latency values are expressed in
    microseconds, they are converted to milliseconds.

    If no structured numeric values are found, the function falls back to
    regular-expression parsing over the serialized payload text.

    Supported units:
    - milliseconds (ms)
    - microseconds (us)

    Args:
        payload (Any): Input payload containing latency information.

    Returns:
        float | None:
            - Latency value in milliseconds.
            - None if no latency value could be extracted.
    """
    if payload is None:
        return None

    numeric_hits = deep_find_numbers(payload, ("latency", "inference_time",))

    for key, value in numeric_hits:
        lower_key = key.lower()

        if "us" in lower_key or "micro" in lower_key:
            return value / 1000.0

        # QAI Hub stores inference_time fields in microseconds (no unit in key name)
        if "inference_time" in lower_key:
            return value / 1000.0

        return value

    text = json.dumps(to_serializable(payload))

    patterns = [
        r"latency[^0-9]*([0-9]+(?:\.[0-9]+)?)\s*ms",
        r"latency[^0-9]*([0-9]+(?:\.[0-9]+)?)\s*us",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:
            value = float(match.group(1))

            return (
                value / 1000.0
                if "us" in pattern
                else value
            )

    return None


def parse_throughput_img_s(payload: Any) -> float | None:
    """
    Extract throughput information from a payload in images per second.

    The function searches recursively for numeric values associated with keys
    related to throughput, such as:
    - "throughput"
    - "images_per_second"
    - "img_s"

    If no structured numeric values are found, the function falls back to
    regular-expression parsing over the serialized payload text.

    Args:
        payload (Any): Input payload containing throughput information.

    Returns:
        float | None:
            - Throughput value in images per second.
            - None if no throughput value could be extracted.
    """
    if payload is None:
        return None

    numeric_hits = deep_find_numbers(
        payload,
        ("throughput", "images_per_second", "img_s"),
    )

    if numeric_hits:
        return numeric_hits[0][1]

    text = json.dumps(to_serializable(payload))

    match = re.search(
        r"throughput[^0-9]*([0-9]+(?:\.[0-9]+)?)",
        text,
        flags=re.IGNORECASE,
    )

    return float(match.group(1)) if match else None


def parse_runtime_name(payload: Any) -> str | None:
    """
    Extract the runtime or backend name from a payload.

    The function serializes the payload into text and searches for known
    runtime identifiers commonly used in inference and deployment pipelines.

    Supported runtime names:
    - CPU
    - GPU
    - HTP
    - DSP
    - QNN

    Matching is case-insensitive.

    Args:
        payload (Any): Input payload containing runtime information.

    Returns:
        str | None:
            - Normalized runtime name in uppercase.
            - None if no known runtime identifier is found.
    """
    if payload is None:
        return None

    text = json.dumps(to_serializable(payload))

    for candidate in ["cpu", "gpu", "htp", "dsp", "qnn"]:
        if candidate in text.lower():
            return candidate.upper()

    return None

def qaihub_job_id(job: Any) -> str | None:
    """
    Extract the job identifier from a Qualcomm AI Hub job object.

    The function checks multiple commonly used attribute names to maximize
    compatibility with different job object implementations.

    Supported attribute names:
    - "job_id"
    - "id"
    - "jobId"

    Args:
        job (Any): Qualcomm AI Hub job object or similar structure.

    Returns:
        str | None:
            - The extracted job identifier as a string.
            - None if no valid job identifier is found.
    """
    for attr in ["job_id", "id", "jobId"]:
        if hasattr(job, attr):
            value = getattr(job, attr)

            if value:
                return str(value)

    return None


def qaihub_job_status(job: Any) -> str:
    """
    Extract the status of a Qualcomm AI Hub job object.

    The function first tries to call ``get_status()`` (the canonical QAI Hub
    SDK API), then falls back to checking ``status`` and ``state`` attributes
    for compatibility with other object shapes.  When the resolved value
    exposes a ``code`` property (e.g. ``JobStatus.code``) the property is
    used so the return value is a plain status name such as ``"SUCCESS"`` or
    ``"FAILED"`` rather than a full object repr.

    Args:
        job (Any): Qualcomm AI Hub job object or similar structure.

    Returns:
        str:
            - The job status code as a plain string (e.g. ``"SUCCESS"``).
            - ``"UNKNOWN"`` if no valid status can be resolved.
    """
    for attr in ["get_status", "status", "state"]:
        if not hasattr(job, attr):
            continue

        value = getattr(job, attr)

        if callable(value):
            try:
                value = value()
            except Exception:
                continue

        if value is None:
            continue

        if hasattr(value, "code"):
            return str(value.code)

        return str(value)

    return "UNKNOWN"


def qaihub_job_url(job: Any) -> str | None:
    """
    Extract the web URL or identifier associated with a Qualcomm AI Hub job.

    The function checks multiple commonly used attribute names to maximize
    compatibility with different job object implementations.

    Supported attribute names:
    - "url"
    - "job_url"
    - "web_url"

    If no URL attribute is found, the function falls back to the job ID.

    Args:
        job (Any): Qualcomm AI Hub job object or similar structure.

    Returns:
        str | None:
            - The extracted job URL or job identifier.
            - None if neither a URL nor a job ID is available.
    """
    for attr in ["url", "job_url", "web_url"]:
        if hasattr(job, attr):
            value = getattr(job, attr)

            if value:
                return str(value)

    job_id = qaihub_job_id(job)

    return job_id


def upload_qaihub_model(client: Any, model: Any) -> Any:
    """
    Upload a model file to Qualcomm AI Hub if a filesystem path is provided.

    If the input model is a string or Path object, the function verifies that
    the file exists and uploads it using the provided Qualcomm AI Hub client.
    Otherwise, the input object is returned unchanged.

    Args:
        client (Any): Qualcomm AI Hub client instance.
        model (Any): Model path or already-uploaded model object.

    Returns:
        Any:
            - Uploaded Qualcomm AI Hub model object if a path is provided.
            - Original input object if it is not a path-like object.

    Raises:
        FileNotFoundError: If the provided model path does not exist.
    """
    if isinstance(model, (str, Path)):
        path = Path(model)

        if not path.exists():
            raise FileNotFoundError(path)

        return client.upload_model(str(path))

    return model


def collect_profile_payload(
    job: Any,
    destination_dir: Path,
) -> Any:
    """
    Collect and persist profiling data from a Qualcomm AI Hub job.

    The function attempts to:
    - Download a binary profile artifact if supported.
    - Retrieve profiling or metrics payloads using common method names.
    - Save the retrieved payload as JSON.
    - Fall back to storing the string representation of the job object if
      no payload is available.

    Supported profiling accessors:
    - download_profile()
    - get_profile()
    - profile
    - get_metrics()
    - get_profile_data()

    Args:
        job (Any): Qualcomm AI Hub job object containing profiling data.
        destination_dir (Path): Directory where profiling artifacts and
            metadata will be stored.

    Returns:
        Any:
            - Retrieved profiling payload if available.
            - None if no payload could be extracted.
    """
    destination_dir.mkdir(parents=True, exist_ok=True)

    payload: Any = None

    if hasattr(job, "download_profile"):
        try:
            target = destination_dir / "profile.bin"
            job.download_profile(str(target))
        except Exception:
            pass
        # QAI Hub SDK writes JSON data alongside the binary (appends .json)
        _bin_json = Path(str(target) + ".json")
        if payload is None and _bin_json.exists():
            try:
                with _bin_json.open(encoding="utf-8") as _fh:
                    payload = json.load(_fh)
            except Exception:
                pass

    for method_name in [
        "get_profile",
        "profile",
        "get_metrics",
        "get_profile_data",
    ]:
        if hasattr(job, method_name):
            method = getattr(job, method_name)

            try:
                payload = method() if callable(method) else method
                break
            except Exception:
                continue

    if payload is not None:
        save_json(
            destination_dir / "profile_payload.json",
            payload,
        )
    else:
        (destination_dir / "profile_payload.txt").write_text(
            str(job),
            encoding="utf-8",
        )

    return payload


def profile_model_qaihub(
    client: Any,
    model: Any,
    device: Any,
    model_name: str,
    backend: str,
    options: str | None = None,
) -> dict[str, Any]:
    """
    Submit a model profiling job to Qualcomm AI Hub and collect its results.

    This function looks up the model specification, prepares a dedicated
    artifact directory, submits a profiling job for the selected backend, waits
    for completion, collects the profiling payload, and extracts key metrics
    such as latency, throughput, and runtime.

    If the QAI Hub client or target device is unavailable, the profiling job is
    skipped. If profiling or parsing fails, the returned result dictionary is
    updated with the corresponding status and notes.

    Args:
        client (Any): Qualcomm AI Hub client instance.
        model (Any): Model path, uploaded model object, or QAI Hub model object.
        device (Any): Target QAI Hub device used for profiling.
        model_name (str): Name of the model as defined in `MODEL_SPECS`.
        backend (str): Backend or compute unit to use for profiling.
        options (str | None, optional): Custom QAI Hub profiling options. If
            None, backend-specific options are generated automatically.
            Defaults to None.

    Returns:
        dict[str, Any]: Profiling summary containing model metadata, backend,
        latency, throughput, runtime, device, job status, job identifier, and
        notes.
    """
    spec_row = next(
        item for item in MODEL_SPECS
        if item["Model"] == model_name
    )

    artifact_dir = (
        PROFILE_ARTIFACTS_DIR
        / sanitize_name(model_name)
        / sanitize_name(backend)
    )
    artifact_dir.mkdir(parents=True, exist_ok=True)

    result: dict[str, Any] = {
        "Model": model_name,
        "Format": spec_row["Format"],
        "Precision": spec_row["Precision"],
        "Pipeline": spec_row["Pipeline"],
        "Backend": backend,
        "Latency_ms": None,
        "Throughput_img_s": None,
        "Runtime": None,
        "Device": str(device) if device is not None else None,
        "Status": "NOT_SUBMITTED",
        "Job_ID": None,
        "Notes": None,
    }

    cached_bin_json_path = artifact_dir / "profile.bin.json"
    cached_json_path = artifact_dir / "profile_payload.json"
    cached_txt_path = artifact_dir / "profile_payload.txt"
    _profile_failed_marker = artifact_dir / "_failed.json"
    if not _profile_failed_marker.exists():
        if cached_bin_json_path.exists() or cached_json_path.exists():
            _cache_src = cached_bin_json_path if cached_bin_json_path.exists() else cached_json_path
            with _cache_src.open(encoding="utf-8") as _fh:
                _cached_payload = json.load(_fh)
            result["Latency_ms"] = parse_latency_ms(_cached_payload)
            result["Throughput_img_s"] = parse_throughput_img_s(_cached_payload)
            if result["Throughput_img_s"] is None and result["Latency_ms"]:
                result["Throughput_img_s"] = round(1000.0 / result["Latency_ms"], 3)
            result["Runtime"] = parse_runtime_name(_cached_payload) or backend.upper()
            result["Status"] = "CACHED"
            if cached_txt_path.exists():
                _txt = cached_txt_path.read_text(encoding="utf-8")
                _id_match = re.search(r'job_id=([a-z0-9]+)', _txt)
                if _id_match:
                    result["Job_ID"] = _id_match.group(1)
            return result
        elif cached_txt_path.exists():
            _cached_text = cached_txt_path.read_text(encoding="utf-8")
            result["Latency_ms"] = parse_latency_ms(_cached_text)
            result["Throughput_img_s"] = parse_throughput_img_s(_cached_text)
            result["Runtime"] = parse_runtime_name(_cached_text) or backend.upper()
            result["Status"] = "CACHED"
            _id_match = re.search(r'job_id=([a-z0-9]+)', _cached_text)
            if _id_match:
                result["Job_ID"] = _id_match.group(1)
            return result

    if client is None or device is None:
        result["Status"] = "SKIPPED"
        result["Notes"] = "QAI Hub client or device is unavailable."
        return result

    profile_options = options or qaihub_backend_option(backend)

    try:
        qai_model = upload_qaihub_model(client, model)

        submit_kwargs: dict[str, Any] = {
            "model": qai_model,
            "device": device,
            "options": profile_options,
        }

        job = client.submit_profile_job(**submit_kwargs)

        if hasattr(job, "wait"):
            job.wait()

        _job_status = qaihub_job_status(job)
        result["Status"] = _job_status
        result["Job_ID"] = qaihub_job_url(job)

        if "FAIL" in _job_status.upper() or "ERROR" in _job_status.upper():
            result["Notes"] = f"QAI Hub profiling job finished with status: {_job_status}"
            save_json(
                _profile_failed_marker,
                {"status": "FAILED", "notes": result["Notes"]},
            )
            return result

        payload = collect_profile_payload(job, artifact_dir)

        result["Latency_ms"] = parse_latency_ms(payload)
        result["Throughput_img_s"] = parse_throughput_img_s(payload)
        if result["Throughput_img_s"] is None and result["Latency_ms"]:
            result["Throughput_img_s"] = round(1000.0 / result["Latency_ms"], 3)
        result["Runtime"] = parse_runtime_name(payload) or backend.upper()

        if (
            result["Latency_ms"] is None
            and result["Throughput_img_s"] is None
        ):
            result["Notes"] = (
                "Automatic profile parsing failed; raw profile artifact was saved."
            )

        _profile_failed_marker.unlink(missing_ok=True)

    except Exception as exc:
        result["Status"] = "FAILED"
        result["Notes"] = str(exc)
        save_json(
            _profile_failed_marker,
            {"status": "FAILED", "notes": str(exc)},
        )

    return result

In [7]:
client, qaihub_client_notes = get_qaihub_client()
print("\n".join(qaihub_client_notes))

profile_requests: list[tuple[str, Path, str]] = []
for spec in MODEL_SPECS:
    path = Path(spec["Path"])
    if not path.exists():
        continue

    if spec["Precision"] == "FP32":
        backends = ["CPU", "GPU"]
    else:
        backends = ["CPU", "HTP/DSP"]

    for backend in backends:
        profile_requests.append((spec["Model"], path, backend))

profile_results = [
    profile_model_qaihub(
        client=client,
        model=path,
        device=DEVICE,
        model_name=model_name,
        backend=backend,
    )
    for model_name, path, backend in tqdm(profile_requests, desc="Profiling with QAI Hub")
]

profile_results_df = pd.DataFrame(
    profile_results,
    columns=[
        "Model",
        "Format",
        "Precision",
        "Pipeline",
        "Backend",
        "Latency_ms",
        "Throughput_img_s",
        "Runtime",
        "Device",
        "Status",
        "Job_ID",
        "Notes",
    ],
)

profile_results_df

qai-hub CLI configured.


Profiling with QAI Hub:   0%|          | 0/12 [00:00<?, ?it/s]

,Model,Format,Precision,Pipeline,Backend,Latency_ms,Throughput_img_s,Runtime,Device,Status,Job_ID,Notes
0,LibreYOLOXs QAI Hub FP32 DLC,DLC,FP32,QAI Hub,CPU,635.783,1.573,CPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jgzvw27op,None
1,LibreYOLOXs QAI Hub FP32 DLC,DLC,FP32,QAI Hub,GPU,207.754,4.813,GPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jpvz4qzkg,None
2,LibreYOLOXs QAI Hub INT8 DLC,DLC,INT8,QAI Hub,CPU,179.081,5.584,CPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jg99808wg,None
3,LibreYOLOXs QAI Hub INT8 DLC,DLC,INT8,QAI Hub,HTP/DSP,6.817,146.692,HTP/DSP,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jp8w828op,None
4,LibreYOLOXs QAIRT FP32 DLC,DLC,FP32,QAIRT,CPU,634.662,1.576,CPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jp1q3277g,None
5,LibreYOLOXs QAIRT FP32 DLC,DLC,FP32,QAIRT,GPU,208.141,4.804,GPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jp3q8nwn5,None
6,LibreYOLOXs QAIRT INT8 DLC,DLC,INT8,QAIRT,CPU,179.974,5.556,CPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jp23jxvqg,None
7,LibreYOLOXs QAIRT INT8 DLC,DLC,INT8,QAIRT,HTP/DSP,7.061,141.623,HTP/DSP,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jgnrx0kq5,None
8,LibreYOLOXs SNPE FP32 DLC,DLC,FP32,SNPE,CPU,640.581,1.561,CPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jg9980mmg,None
9,LibreYOLOXs SNPE FP32 DLC,DLC,FP32,SNPE,GPU,212.380,4.709,GPU,"Device(name='Dragonwing RB3 Gen 2 Vision Kit',...",CACHED,jp23jxr6g,None


## Local QAIRT / SNPE Inspection

QAI Hub is the primary benchmarking service, but local QAIRT and SNPE CLIs remain useful for **structural inspection** of DLC artifacts.

The following cells:

- run `qairt-dlc-info --input_dlc <model.dlc>`
- run `snpe-dlc-info --input_dlc <model.dlc>`
- save the outputs to `reports/dlc_info/`

This section is intentionally inspection-only; it does not replace the QAI Hub benchmarking path.

In [8]:
def run_command_safely(
    command: list[str],
) -> tuple[int, str, str]:
    """
    Execute a shell command safely and capture its results.

    The function executes a subprocess command without raising exceptions for
    non-zero exit codes. It captures standard output and standard error as text
    and handles common execution errors gracefully.

    Error handling:
    - FileNotFoundError -> return code 127
    - Any other exception -> return code 1

    Args:
        command (list[str]): Command and arguments to execute.

    Returns:
        tuple[int, str, str]:
            - Process return code.
            - Captured standard output.
            - Captured standard error or exception message.
    """
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
        )

        return (
            completed.returncode,
            completed.stdout,
            completed.stderr,
        )

    except FileNotFoundError as exc:
        return 127, "", str(exc)

    except Exception as exc:
        return 1, "", str(exc)


def inspect_dlc(
    model_path: Path,
    tool_name: str,
) -> dict[str, Any]:
    """
    Inspect a DLC model using an external Qualcomm inspection tool.

    The function executes a command-line inspection tool (such as
    `qairt-dlc-info` or `snpe-dlc-info`) against the provided DLC model file,
    captures the command output, and stores a detailed inspection report.

    If the model file does not exist, a report describing the missing file is
    generated instead.

    Args:
        model_path (Path): Path to the DLC model file.
        tool_name (str): Name of the inspection tool executable.

    Returns:
        dict[str, Any]: Dictionary containing:
            - "Tool" (str): Inspection tool name.
            - "Model_Path" (str): DLC model path.
            - "Return_Code" (int): Command return code.
            - "Status" (str): Inspection status.
            - "Report_Path" (str): Path to the generated report file.
    """
    report_name = (
        f"{sanitize_name(tool_name)}__"
        f"{sanitize_name(model_path.stem)}.txt"
    )

    report_path = DLC_INFO_DIR / report_name

    if not model_path.exists():
        report_path.write_text(
            f"Missing model file: {model_path}\n",
            encoding="utf-8",
        )

        return {
            "Tool": tool_name,
            "Model_Path": str(model_path),
            "Return_Code": 1,
            "Status": "MISSING_MODEL",
            "Report_Path": str(report_path),
        }

    return_code, stdout, stderr = run_command_safely(
        [tool_name, "--input_dlc", str(model_path)]
    )

    report_path.write_text(
        f"$ {' '.join([tool_name, '--input_dlc', str(model_path)])}\n\n"
        f"Return code: {return_code}\n\n"
        f"STDOUT\n{'-' * 80}\n{stdout}\n\n"
        f"STDERR\n{'-' * 80}\n{stderr}\n",
        encoding="utf-8",
    )

    return {
        "Tool": tool_name,
        "Model_Path": str(model_path),
        "Return_Code": return_code,
        "Status": "OK" if return_code == 0 else "FAILED",
        "Report_Path": str(report_path),
    }


inspection_jobs = [
    (QAIRT_FP32_DLC, "qairt-dlc-info"),
    (QAIRT_INT8_DLC, "qairt-dlc-info"),
    (SNPE_FP32_DLC, "snpe-dlc-info"),
    (SNPE_INT8_DLC, "snpe-dlc-info"),
]

dlc_inspection_df = pd.DataFrame(
    [inspect_dlc(model_path, tool_name) for model_path, tool_name in inspection_jobs]
)

dlc_inspection_df

,Tool,Model_Path,Return_Code,Status,Report_Path
0,qairt-dlc-info,../models/qairt/LibreYOLOXs_fp32.dlc,0,OK,reports/dlc_info/qairt-dlc-info__LibreYOLOXs_f...
1,qairt-dlc-info,../models/qairt/LibreYOLOXs_int8.dlc,0,OK,reports/dlc_info/qairt-dlc-info__LibreYOLOXs_i...
2,snpe-dlc-info,../models/snpe/LibreYOLOXs_fp32.dlc,0,OK,reports/dlc_info/snpe-dlc-info__LibreYOLOXs_fp...
3,snpe-dlc-info,../models/snpe/LibreYOLOXs_int8.dlc,0,OK,reports/dlc_info/snpe-dlc-info__LibreYOLOXs_in...


## QAI Hub Inference Jobs

We now run inference on the validation subset. The same validation inputs are reused across candidate models so that later comparisons remain aligned.

**Practical note:** a full validation subset of 100 tensors can be memory-heavy because each input is `(1, 3, 640, 640)` float32. The notebook keeps this configurable through `INFERENCE_SAMPLE_LIMIT`.

In [9]:
def normalize_inference_outputs(
    payload: Any,
    num_samples: int | None = None,
) -> dict[str, list[np.ndarray]]:
    """
    Normalize inference outputs into a standardized dictionary format.

    The function converts different inference output structures into a unified
    dictionary mapping output names to lists of NumPy arrays. This simplifies
    downstream processing and comparison across inference frameworks and
    runtimes.

    Supported input formats:
    - Dictionary of outputs
    - List of dictionaries
    - List of arrays
    - Single array-like object

    If `num_samples` is provided and the first dimension of an output matches
    the number of samples, the output is split into one array per sample.

    Args:
        payload (Any): Raw inference output payload.
        num_samples (int | None, optional): Number of samples contained in the
            batch dimension. If provided, batched outputs are separated into
            individual sample arrays. Defaults to None.

    Returns:
        dict[str, list[np.ndarray]]:
            Dictionary where:
            - Keys are normalized output names.
            - Values are lists of NumPy arrays representing outputs per sample.
    """
    outputs: dict[str, list[np.ndarray]] = {}

    if payload is None:
        return outputs

    if isinstance(payload, dict):
        for key, value in payload.items():
            if isinstance(value, list):
                # QAI Hub DatasetEntries: value is already a per-sample list
                # of arrays (e.g. [arr_(1,8400,4), ...]). Use them directly
                # to avoid np.asarray stacking them into (N, 1, 8400, 4).
                outputs[str(key)] = [np.asarray(item) for item in value]
            else:
                array = np.asarray(value)
                if (
                    num_samples
                    and array.ndim >= 1
                    and array.shape[0] == num_samples
                ):
                    outputs[str(key)] = [
                        np.asarray(array[index:index + 1])
                        for index in range(num_samples)
                    ]
                else:
                    outputs[str(key)] = [array]

        return outputs

    if isinstance(payload, list):
        if payload and isinstance(payload[0], dict):
            keys = sorted(payload[0].keys())

            for key in keys:
                outputs[str(key)] = [
                    np.asarray(item[key])
                    for item in payload
                ]

            return outputs

        # Each element is a separate output tensor (e.g. one per model output).
        # Assign them to distinct keys so downstream decoders can distinguish
        # bboxes from scores instead of merging everything under "output_0".
        for index, item in enumerate(payload):
            key = f"output_{index}"
            array = np.asarray(item)

            if (
                num_samples
                and array.ndim >= 1
                and array.shape[0] == num_samples
            ):
                outputs[key] = [
                    np.asarray(array[i:i + 1])
                    for i in range(num_samples)
                ]
            else:
                outputs[key] = [array]

        return outputs

    array = np.asarray(payload)

    if (
        num_samples
        and array.ndim >= 1
        and array.shape[0] == num_samples
    ):
        outputs["output_0"] = [
            np.asarray(array[index:index + 1])
            for index in range(num_samples)
        ]
    else:
        outputs["output_0"] = [array]

    return outputs


def collect_inference_payload(
    job: Any,
    destination_dir: Path,
    num_samples: int | None = None,
) -> dict[str, list[np.ndarray]]:
    """
    Collect and normalize inference outputs from a Qualcomm AI Hub job.

    The function attempts to retrieve inference outputs using multiple
    supported APIs, optionally downloads output artifacts, stores the raw
    payload as JSON, and converts the outputs into a normalized structure for
    downstream processing.

    Supported output accessors:
    - download_output_data()
    - get_output_data()
    - get_outputs()
    - outputs
    - output_data

    Args:
        job (Any): Qualcomm AI Hub inference job object.
        destination_dir (Path): Directory where output artifacts and payloads
            will be stored.
        num_samples (int | None, optional): Number of samples in the batch
            dimension. If provided, outputs are split into per-sample arrays.
            Defaults to None.

    Returns:
        dict[str, list[np.ndarray]]:
            Dictionary containing normalized inference outputs where:
            - Keys are output names.
            - Values are lists of NumPy arrays.
    """
    destination_dir.mkdir(parents=True, exist_ok=True)

    payload: Any = None

    if hasattr(job, "download_output_data"):
        try:
            # Save artifact to disk, then immediately retrieve the in-memory
            # DatasetEntries (Mapping[str, list[np.ndarray]]) so that the
            # fallback loop below never receives a raw Dataset object.
            job.download_output_data(str(destination_dir))
            try:
                payload = job.download_output_data()
            except Exception:
                pass

        except TypeError:
            try:
                payload = job.download_output_data()
            except Exception:
                pass

        except Exception:
            pass

    for method_name in [
        "get_output_data",
        "get_outputs",
        "outputs",
        "output_data",
    ]:
        if payload is not None:
            break

        if hasattr(job, method_name):
            method = getattr(job, method_name)

            try:
                payload = (
                    method()
                    if callable(method)
                    else method
                )
            except Exception:
                continue

    # Unwrap Dataset-like objects (e.g. qai_hub.Dataset) into DatasetEntries.
    # download_output_data(dir) populates job.outputs with a Dataset object;
    # calling .download() with no arguments converts it to
    # Mapping[str, list[np.ndarray]] which normalize_inference_outputs expects.
    if (
        payload is not None
        and not isinstance(payload, (dict, list, np.ndarray))
        and hasattr(payload, "download")
        and callable(getattr(payload, "download"))
    ):
        try:
            payload = payload.download()
        except Exception:
            payload = None

    if payload is not None:
        save_json(
            destination_dir / "raw_output_payload.json",
            payload,
        )

    return normalize_inference_outputs(
        payload,
        num_samples=num_samples,
    )


def save_outputs_bundle(
    outputs: dict[str, list[np.ndarray]],
    destination_dir: Path,
) -> None:
    """
    Save normalized inference outputs as NumPy files with a JSON manifest.

    Each output array is saved as a separate `.npy` file. A manifest file is
    also generated to describe the saved outputs, including their paths, shapes,
    and data types.

    Args:
        outputs (dict[str, list[np.ndarray]]): Dictionary mapping output names
            to lists of NumPy arrays.
        destination_dir (Path): Directory where output files and the manifest
            will be saved.

    Returns:
        None
    """
    destination_dir.mkdir(parents=True, exist_ok=True)

    summary: dict[str, Any] = {}

    for output_name, arrays in outputs.items():
        summary[output_name] = []

        for index, array in enumerate(arrays):
            array_np = np.asarray(array)
            array_path = (
                destination_dir
                / f"{sanitize_name(output_name)}_{index:04d}.npy"
            )

            np.save(array_path, array_np)

            summary[output_name].append(
                {
                    "path": str(array_path),
                    "shape": list(array_np.shape),
                    "dtype": str(array_np.dtype),
                }
            )

    save_json(destination_dir / "manifest.json", summary)


def load_outputs_bundle(
    source_dir: Path
) -> dict[str, list[np.ndarray]]:
    """
    Load saved inference outputs from a directory previously written by
    ``save_outputs_bundle``.

    Args:
        source_dir (Path): Directory containing ``manifest.json`` and the
            corresponding ``.npy`` array files.

    Returns:
        dict[str, list[np.ndarray]]: Mapping from output name to a list of
        NumPy arrays, matching the format produced by
        ``normalize_inference_outputs``.
    """
    manifest_path = source_dir / "manifest.json"
    if not manifest_path.exists():
        return {}
    with manifest_path.open(encoding="utf-8") as _fh:
        _manifest = json.load(_fh)
    _outputs: dict[str, list[np.ndarray]] = {}
    for _output_name, _entries in _manifest.items():
        _arrays: list[np.ndarray] = []
        for _entry in _entries:
            _array_path = Path(_entry["path"])
            if _array_path.exists():
                _arrays.append(np.load(_array_path, allow_pickle=True))
        _outputs[_output_name] = _arrays
    return _outputs


def run_inference_qaihub(
    client: Any,
    model: Any,
    device: Any,
    inputs: dict[str, list[np.ndarray]],
    model_name: str,
    backend: str = "cpu",
) -> dict[str, Any]:
    """
    Run model inference through Qualcomm AI Hub and persist the outputs.

    The function uploads the model if needed, submits an inference job to
    Qualcomm AI Hub, waits for completion, collects the output payload, saves
    normalized outputs to disk, and returns a structured execution summary.

    It also supports cached outputs by loading a previously saved output bundle
    when a manifest exists and no failure marker is present.

    Args:
        client (Any): Qualcomm AI Hub client instance.
        model (Any): Model path, uploaded model object, or QAI Hub model object.
        device (Any): Target QAI Hub device used for inference.
        inputs (dict[str, list[np.ndarray]]): Input tensors organized by input
            name. The function expects an `"images"` key for image tensors.
        model_name (str): Human-readable model name used for output directory
            naming and reporting.
        backend (str): Compute unit to use for inference. Converted to a
            QAI Hub ``--compute_unit`` option via ``qaihub_backend_option``.
            Defaults to ``"cpu"``.

    Returns:
        dict[str, Any]: Dictionary containing:
            - "Model" (str): Model name.
            - "Status" (str): Inference status.
            - "Outputs" (dict[str, list[np.ndarray]]): Normalized outputs.
            - "Job_ID" (str | None): QAI Hub job URL or identifier.
            - "Notes" (str | None): Additional status or error details.
    """
    if client is None or device is None:
        return {
            "Model": model_name,
            "Status": "SKIPPED",
            "Outputs": {},
            "Notes": "QAI Hub client or device is unavailable.",
        }

    destination_dir = (
        INFERENCE_OUTPUTS_DIR
        / sanitize_name(model_name)
        / sanitize_name(backend)
    )
    destination_dir.mkdir(parents=True, exist_ok=True)

    _manifest_path = destination_dir / "manifest.json"
    _inference_failed_marker = destination_dir / "_failed.json"

    if (
        not _inference_failed_marker.exists()
        and _manifest_path.exists()
    ):
        _cached_outputs = load_outputs_bundle(destination_dir)
        _job_info_path = destination_dir / "job_info.json"
        _cached_job_id = None
        if _job_info_path.exists():
            try:
                with _job_info_path.open(encoding="utf-8") as _fh:
                    _cached_job_id = json.load(_fh).get("job_id")
            except Exception:
                pass

        return {
            "Model": model_name,
            "Status": "CACHED",
            "Outputs": _cached_outputs,
            "Job_ID": _cached_job_id,
            "Notes": None,
        }

    try:
        qai_model = upload_qaihub_model(client, model)

        inference_options = qaihub_backend_option(backend)

        submit_kwargs: dict[str, Any] = {
            "model": qai_model,
            "device": device,
            "inputs": inputs,
            "options": inference_options,
        }

        try:
            job = client.submit_inference_job(**submit_kwargs)
        except TypeError:
            submit_kwargs["input_data"] = submit_kwargs.pop("inputs")
            job = client.submit_inference_job(**submit_kwargs)

        if hasattr(job, "wait"):
            job.wait()

        _inf_job_status = qaihub_job_status(job)

        if (
            "FAIL" in _inf_job_status.upper()
            or "ERROR" in _inf_job_status.upper()
        ):
            _inf_notes = (
                f"QAI Hub inference job finished with status: "
                f"{_inf_job_status}"
            )

            save_json(
                _inference_failed_marker,
                {
                    "status": "FAILED",
                    "notes": _inf_notes,
                },
            )

            return {
                "Model": model_name,
                "Status": _inf_job_status,
                "Outputs": {},
                "Job_ID": qaihub_job_url(job),
                "Notes": _inf_notes,
            }

        outputs = collect_inference_payload(
            job,
            destination_dir=destination_dir,
            num_samples=len(inputs.get("images", [])),
        )

        save_outputs_bundle(outputs, destination_dir)
        _inference_failed_marker.unlink(missing_ok=True)
        _live_job_id = qaihub_job_url(job)
        save_json(
            destination_dir / "job_info.json",
            {"job_id": _live_job_id, "status": _inf_job_status},
        )

        return {
            "Model": model_name,
            "Status": _inf_job_status,
            "Outputs": outputs,
            "Job_ID": _live_job_id,
            "Notes": (
                None
                if outputs
                else "Inference job completed but no outputs were parsed automatically."
            ),
        }

    except Exception as exc:
        save_json(
            _inference_failed_marker,
            {
                "status": "FAILED",
                "notes": str(exc),
            },
        )

        return {
            "Model": model_name,
            "Status": "FAILED",
            "Outputs": {},
            "Job_ID": None,
            "Notes": str(exc),
        }


def load_validation_raw_paths() -> list[Path]:
    """
    Load validation RAW file paths from the validation manifest file.

    The function reads the `filenames.txt` manifest located in the validation
    directory and returns all non-empty paths as `Path` objects.

    Returns:
        list[Path]:
            List of validation RAW file paths. Returns an empty list if the
            manifest file does not exist.
    """
    manifest_path = VAL_DIR / "filenames.txt"

    if not manifest_path.exists():
        return []

    return [
        Path(line.strip())
        for line in manifest_path.read_text().splitlines()
        if line.strip()
    ]

In [10]:
INFERENCE_SAMPLE_LIMIT = None  # Set to an integer for a shorter run.

raw_files = load_validation_raw_paths()
if INFERENCE_SAMPLE_LIMIT is not None:
    raw_files = raw_files[:INFERENCE_SAMPLE_LIMIT]

validation_inputs = {
    "images": [
        np.fromfile(path, dtype=np.float32).reshape(1, 3, 640, 640)
        for path in raw_files
        if path.exists()
    ]
}

inference_candidates = [
    spec for spec in MODEL_SPECS
    if Path(spec["Path"]).exists()
]

inference_requests: list[tuple[str, Path, str]] = []
for spec in inference_candidates:
    backends = ["CPU", "GPU"] if spec["Precision"] == "FP32" else ["CPU", "HTP/DSP"]
    for backend in backends:
        inference_requests.append((spec["Model"], Path(spec["Path"]), backend))

inference_results: dict[tuple[str, str], dict] = {
    (model_name, backend): run_inference_qaihub(
        client=client,
        model=path,
        device=DEVICE,
        inputs=validation_inputs,
        model_name=model_name,
        backend=backend,
    )
    for model_name, path, backend in tqdm(inference_requests, desc="Running inference")
}

pd.DataFrame(
    [
        {
            "Model": result["Model"],
            "Backend": backend,
            "Status": result["Status"],
            "Job_ID": result.get("Job_ID"),
            "Notes": result.get("Notes"),
        }
        for (model_name, backend), result in inference_results.items()
    ]
)

Running inference:   0%|          | 0/12 [00:00<?, ?it/s]

,Model,Backend,Status,Job_ID,Notes
0,LibreYOLOXs QAI Hub FP32 DLC,CPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jg99...,None
1,LibreYOLOXs QAI Hub FP32 DLC,GPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jp0e...,None
2,LibreYOLOXs QAI Hub INT8 DLC,CPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jgnr...,None
3,LibreYOLOXs QAI Hub INT8 DLC,HTP/DSP,CACHED,https://workbench.aihub.qualcomm.com/jobs/jpe4...,None
4,LibreYOLOXs QAIRT FP32 DLC,CPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jp8w...,None
5,LibreYOLOXs QAIRT FP32 DLC,GPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/j57v...,None
6,LibreYOLOXs QAIRT INT8 DLC,CPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jgjk...,None
7,LibreYOLOXs QAIRT INT8 DLC,HTP/DSP,CACHED,https://workbench.aihub.qualcomm.com/jobs/jgd7...,None
8,LibreYOLOXs SNPE FP32 DLC,CPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/j56q...,None
9,LibreYOLOXs SNPE FP32 DLC,GPU,CACHED,https://workbench.aihub.qualcomm.com/jobs/jp3q...,None


## Output Similarity Analysis

Quantization should ideally preserve output structure and ranking even when the model becomes smaller and faster. We therefore compare outputs using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Max Absolute Error
- Mean Relative Error
- Cosine Similarity
- Pearson Correlation

The notebook is written to support multiple pipeline pairs if outputs are available. Therefore, the **QAI Hub FP32 DLC vs QAI Hub INT8 DLC** pair is treated as the primary reference comparison.

In [11]:
def flatten_output_list(
    output_list: list[np.ndarray],
) -> np.ndarray:
    """
    Flatten and concatenate a list of NumPy arrays into a single 1D array.

    Each array is converted to `float64`, flattened, and concatenated into a
    single continuous NumPy array. If the input list is empty, an empty
    float32 array is returned.

    Args:
        output_list (list[np.ndarray]): List of NumPy arrays to flatten and
            concatenate.

    Returns:
        np.ndarray:
            One-dimensional concatenated NumPy array containing all flattened
            values.
    """
    if not output_list:
        return np.array([], dtype=np.float32)

    return np.concatenate(
        [
            np.asarray(item, dtype=np.float64).reshape(-1)
            for item in output_list
        ]
    )


def compute_output_similarity(
    reference_outputs: Any,
    candidate_outputs: Any,
) -> pd.DataFrame:
    """
    Compute similarity metrics between reference and candidate inference outputs.

    The function normalizes both output payloads, identifies common output names,
    flattens matching outputs, aligns them by the shortest available length, and
    computes numerical difference and similarity metrics.

    Args:
        reference_outputs (Any): Reference inference outputs.
        candidate_outputs (Any): Candidate inference outputs to compare against
            the reference outputs.

    Returns:
        pd.DataFrame: DataFrame containing one row per common output with:
            - Output_Name
            - MAE
            - RMSE
            - Max_Abs_Error
            - Mean_Relative_Error
            - Cosine_Similarity
            - Pearson_Correlation
    """
    records: list[dict[str, Any]] = []

    ref_dict = normalize_inference_outputs(reference_outputs)
    cand_dict = normalize_inference_outputs(candidate_outputs)

    common_outputs = sorted(set(ref_dict) & set(cand_dict))

    for output_name in common_outputs:
        ref_flat = flatten_output_list(ref_dict[output_name])
        cand_flat = flatten_output_list(cand_dict[output_name])

        if ref_flat.size == 0 or cand_flat.size == 0:
            continue

        length = min(ref_flat.size, cand_flat.size)
        ref_flat = ref_flat[:length]
        cand_flat = cand_flat[:length]

        diff = cand_flat - ref_flat

        mae = float(np.mean(np.abs(diff)))
        rmse = float(np.sqrt(np.mean(np.square(diff))))
        max_abs = float(np.max(np.abs(diff)))
        mean_relative = float(
            np.mean(
                np.abs(diff)
                / np.maximum(np.abs(ref_flat), 1e-12)
            )
        )

        ref_norm = float(np.linalg.norm(ref_flat))
        cand_norm = float(np.linalg.norm(cand_flat))

        cosine_similarity: float | None = None
        if ref_norm > 0 and cand_norm > 0:
            cosine_similarity = float(
                np.dot(ref_flat, cand_flat)
                / (ref_norm * cand_norm)
            )

        pearson_correlation: float | None = None
        if np.std(ref_flat) > 0 and np.std(cand_flat) > 0:
            pearson_correlation = float(
                np.corrcoef(ref_flat, cand_flat)[0, 1]
            )

        records.append(
            {
                "Output_Name": output_name,
                "MAE": mae,
                "RMSE": rmse,
                "Max_Abs_Error": max_abs,
                "Mean_Relative_Error": mean_relative,
                "Cosine_Similarity": cosine_similarity,
                "Pearson_Correlation": pearson_correlation,
            }
        )

    return pd.DataFrame(records)

In [12]:
similarity_pairs = [
    ("QAI Hub", "LibreYOLOXs QAI Hub FP32 DLC", "LibreYOLOXs QAI Hub INT8 DLC"),
    ("QAIRT", "LibreYOLOXs QAIRT FP32 DLC", "LibreYOLOXs QAIRT INT8 DLC"),
    ("SNPE", "LibreYOLOXs SNPE FP32 DLC", "LibreYOLOXs SNPE INT8 DLC"),
]

similarity_frames = []
for pipeline_name, reference_model, candidate_model in similarity_pairs:
    reference_bundle = inference_results.get((reference_model, "CPU"), {}).get("Outputs", {})
    candidate_bundle = inference_results.get((candidate_model, "CPU"), {}).get("Outputs", {})
    if not reference_bundle or not candidate_bundle:
        continue
    frame = compute_output_similarity(reference_bundle, candidate_bundle)
    if frame.empty:
        continue
    frame.insert(0, "Candidate_Model", candidate_model)
    frame.insert(0, "Reference_Model", reference_model)
    frame.insert(0, "Pipeline", pipeline_name)
    similarity_frames.append(frame)

output_similarity_df = (
    pd.concat(similarity_frames, ignore_index=True)
    if similarity_frames
    else pd.DataFrame(
        columns=[
            "Pipeline",
            "Reference_Model",
            "Candidate_Model",
            "Output_Name",
            "MAE",
            "RMSE",
            "Max_Abs_Error",
            "Mean_Relative_Error",
            "Cosine_Similarity",
            "Pearson_Correlation",
        ]
    )
)

output_similarity_df

,Pipeline,Reference_Model,Candidate_Model,Output_Name,MAE,RMSE,Max_Abs_Error,Mean_Relative_Error,Cosine_Similarity,Pearson_Correlation
0,QAI Hub,LibreYOLOXs QAI Hub FP32 DLC,LibreYOLOXs QAI Hub INT8 DLC,bboxes,14.691086,34.453488,626.884542,0.256851,0.991901,0.983349
1,QAI Hub,LibreYOLOXs QAI Hub FP32 DLC,LibreYOLOXs QAI Hub INT8 DLC,scores,0.007802,0.029047,0.999704,733.637206,0.594703,0.577878
2,QAIRT,LibreYOLOXs QAIRT FP32 DLC,LibreYOLOXs QAIRT INT8 DLC,bboxes,16.588699,37.753529,686.889587,0.347785,0.990386,0.980236
3,QAIRT,LibreYOLOXs QAIRT FP32 DLC,LibreYOLOXs QAIRT INT8 DLC,scores,0.008058,0.029646,0.999724,5456.121838,0.577970,0.560684
4,SNPE,LibreYOLOXs SNPE FP32 DLC,LibreYOLOXs SNPE INT8 DLC,bboxes,16.588699,37.753529,686.889587,0.347785,0.990386,0.980236
5,SNPE,LibreYOLOXs SNPE FP32 DLC,LibreYOLOXs SNPE INT8 DLC,scores,0.008058,0.029646,0.999724,5456.121838,0.577970,0.560684


## LibreYOLOXs Post-processing

The exported LibreYOLOXs artifacts are documented to expose:

- `bboxes`: shape `(1, 8400, 4)` with `[cx, cy, w, h]`
- `scores`: shape `(1, 8400, 81)` with objectness + 80 class scores

The post-processing below:

- applies confidence filtering
- performs NMS
- converts to COCO-style predictions
- rescales boxes to original image size using `metadata.csv`
- applies the **top-left letterbox correction** (`pad_x = pad_y = 0`)

**TODO:** if a backend returns a different tensor structure, adapt the decoding logic in `decode_model_outputs()` accordingly.

In [13]:
COCO80_TO_91 = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    22, 23, 24, 25, 27, 28, 31, 32, 33, 34,
    35, 36, 37, 38, 39, 40, 41, 42, 43, 44,
    46, 47, 48, 49, 50, 51, 52, 53, 54, 55,
    56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
    67, 70, 72, 73, 74, 75, 76, 77, 78, 79,
    80, 81, 82, 84, 85, 86, 87, 88, 89, 90,
]


def decode_model_outputs(outputs: Any) -> tuple[np.ndarray, np.ndarray]:
    """
    Decode normalized model outputs into bounding boxes and scores.

    The LibreYOLOXs model exposes two separate output tensors:
    - ``"bboxes"``: shape ``(1, 8400, 4)`` — bounding boxes in
      ``(cx, cy, w, h)`` format.
    - ``"scores"``: shape ``(1, 8400, 81)`` — objectness score at index 0
      followed by 80 COCO class scores.

    When inference backends (e.g. Qualcomm AI Hub) return outputs with
    generic numeric keys (``"0"``, ``"1"``) instead of the original layer
    names, the function falls back to shape-based detection: the array with
    ``shape[-1] == 4`` is treated as bounding boxes and the array with
    ``shape[-1] in (80, 81)`` is treated as scores.

    Args:
        outputs (Any): Raw or normalized model output payload.

    Returns:
        tuple[np.ndarray, np.ndarray]:
            - Bounding boxes array of shape ``(N, 4)``.
            - Scores array of shape ``(N, 80)`` or ``(N, 81)``.

    Raises:
        NotImplementedError: If the output structure cannot be decoded.
    """
    normalized = normalize_inference_outputs(outputs)

    if "bboxes" in normalized and "scores" in normalized:
        bboxes = np.asarray(normalized["bboxes"][0])
        scores = np.asarray(normalized["scores"][0])

        if bboxes.ndim == 3:
            bboxes = bboxes[0]

        if scores.ndim == 3:
            scores = scores[0]

        return bboxes, scores

    # Fallback: backends such as QAI Hub may return outputs with numeric keys
    # ("0", "1") rather than the original layer names. Identify the two tensors
    # by their last-dimension size: 4 → bboxes, 80 or 81 → scores.
    bboxes_candidate: np.ndarray | None = None
    scores_candidate: np.ndarray | None = None

    for key in sorted(normalized.keys()):
        arr = np.asarray(normalized[key][0])

        if arr.ndim == 3:
            arr = arr[0]

        if arr.ndim == 2 and arr.shape[-1] == 4:
            bboxes_candidate = arr
        elif arr.ndim == 2 and arr.shape[-1] in (80, 81):
            scores_candidate = arr

    if bboxes_candidate is not None and scores_candidate is not None:
        return bboxes_candidate, scores_candidate

    shapes = {k: list(np.asarray(v[0]).shape) for k, v in normalized.items()}
    raise NotImplementedError(
        f"decode_model_outputs: unsupported output structure. "
        f"Found keys/shapes: {shapes}"
    )


def apply_nms_xywh(
    boxes_xywh: np.ndarray,
    scores: np.ndarray,
    nms_threshold: float,
) -> list[int]:
    """
    Apply Non-Maximum Suppression (NMS) to bounding boxes in XYWH format.

    The function uses OpenCV's `cv.dnn.NMSBoxes` implementation to suppress
    overlapping bounding boxes based on their confidence scores and the
    specified NMS threshold.

    Args:
        boxes_xywh (np.ndarray): Bounding boxes in `[x, y, width, height]`
            format.
        scores (np.ndarray): Confidence scores associated with each bounding
            box.
        nms_threshold (float): Intersection-over-Union (IoU) threshold used
            for suppression.

    Returns:
        list[int]:
            List of selected bounding box indices after NMS.
    """
    if len(boxes_xywh) == 0:
        return []

    box_list = boxes_xywh.tolist()
    score_list = scores.astype(float).tolist()

    selected = cv.dnn.NMSBoxes(
        box_list,
        score_list,
        score_threshold=0.0,
        nms_threshold=nms_threshold,
    )

    if selected is None or len(selected) == 0:
        return []

    selected = np.asarray(selected).reshape(-1).tolist()

    return [int(index) for index in selected]


def postprocess_outputs(
    outputs: Any,
    metadata_row: dict[str, Any],
    conf_threshold: float = 0.25,
    nms_threshold: float = 0.45,
) -> list[dict[str, Any]]:
    """
    Post-process raw model outputs into COCO-style object detection predictions.

    The function decodes model outputs into bounding boxes and scores, applies
    confidence filtering, rescales boxes back to the original image dimensions,
    applies Non-Maximum Suppression, and returns predictions using COCO category
    IDs.

    Args:
        outputs (Any): Raw or normalized model outputs.
        metadata_row (dict[str, Any]): Metadata for the corresponding image,
            including `image_id`, `orig_w`, `orig_h`, and `ratio`.
        conf_threshold (float, optional): Minimum confidence score required to
            keep a detection. Defaults to 0.25.
        nms_threshold (float, optional): IoU threshold used for Non-Maximum
            Suppression. Defaults to 0.45.

    Returns:
        list[dict[str, Any]]: COCO-style predictions containing `image_id`,
        `category_id`, `bbox`, and `score`.

    Raises:
        ValueError: If the decoded scores array is not rank-2.
        NotImplementedError: If the score layout is not supported.
    """
    bboxes, scores = decode_model_outputs(outputs)

    if scores.ndim != 2:
        raise ValueError(
            f"Expected scores to be rank-2, received shape {scores.shape}"
        )

    if scores.shape[1] == 81:
        objectness = scores[:, 0]
        class_scores = scores[:, 1:]
    elif scores.shape[1] == 80:
        objectness = np.ones(scores.shape[0], dtype=np.float32)
        class_scores = scores
    else:
        raise NotImplementedError(
            "TODO: adapt score decoding for your model output layout."
        )

    class_indices = np.argmax(class_scores, axis=1)
    class_confidences = class_scores[
        np.arange(class_scores.shape[0]),
        class_indices,
    ]
    combined_scores = objectness * class_confidences

    keep = combined_scores >= conf_threshold

    if not np.any(keep):
        return []

    bboxes = bboxes[keep]
    class_indices = class_indices[keep]
    combined_scores = combined_scores[keep]

    cx, cy, width, height = bboxes.T

    x = cx - width / 2.0
    y = cy - height / 2.0

    ratio = float(metadata_row["ratio"])
    orig_w = float(metadata_row["orig_w"])
    orig_h = float(metadata_row["orig_h"])

    x = x / ratio
    y = y / ratio
    width = width / ratio
    height = height / ratio

    x = np.clip(x, 0, orig_w)
    y = np.clip(y, 0, orig_h)
    width = np.clip(width, 0, orig_w - x)
    height = np.clip(height, 0, orig_h - y)

    boxes_xywh = np.stack(
        [x, y, width, height],
        axis=1,
    )

    selected_indices = apply_nms_xywh(
        boxes_xywh,
        combined_scores,
        nms_threshold,
    )

    predictions: list[dict[str, Any]] = []

    for index in selected_indices:
        class_index = int(class_indices[index])

        if class_index >= len(COCO80_TO_91):
            continue

        predictions.append(
            {
                "image_id": int(metadata_row["image_id"]),
                "category_id": int(COCO80_TO_91[class_index]),
                "bbox": [
                    float(value)
                    for value in boxes_xywh[index]
                ],
                "score": float(combined_scores[index]),
            }
        )

    return predictions

## Accuracy / mAP Evaluation

If COCO annotations are available, we compute:

- `mAP@0.5`
- `mAP@0.5:0.95`
- precision
- recall

The metrics are computed locally with `pycocotools`.

In [14]:
def evaluate_coco_map(
    predictions: list[dict[str, Any]],
    annotation_file: Path | str,
) -> dict[str, Any]:
    """
    Evaluate object detection predictions using COCO mAP metrics.

    The function evaluates COCO-style bounding box predictions against a COCO
    annotation file using `pycocotools`. If the required dependency,
    annotation file, or predictions are unavailable, placeholder metrics are
    returned with explanatory notes.

    Args:
        predictions (list[dict[str, Any]]): COCO-style prediction records
            containing `image_id`, `category_id`, `bbox`, and `score`.
        annotation_file (Path | str): Path to the COCO annotation JSON file.

    Returns:
        dict[str, Any]: Evaluation results containing:
            - "mAP@0.5" (float | None): Mean Average Precision at IoU 0.5.
            - "mAP@0.5:0.95" (float | None): COCO-style mAP averaged across
              IoU thresholds from 0.5 to 0.95.
            - "Precision_Score" (float | None): Mean precision score.
            - "Recall" (float | None): Mean recall score.
            - "Notes" (str | None): Additional status or error message.
    """
    placeholder: dict[str, Any] = {
        "mAP@0.5": None,
        "mAP@0.5:0.95": None,
        "Precision_Score": None,
        "Recall": None,
        "Notes": None,
    }

    if not PYCOCOTOOLS_AVAILABLE:
        placeholder["Notes"] = (
            "pycocotools is not installed. Install it with "
            "`pip install pycocotools` and rerun this cell."
        )
        return placeholder

    annotation_path = Path(annotation_file)

    if not annotation_path.exists():
        placeholder["Notes"] = (
            f"Annotation file not found: {annotation_path}"
        )
        return placeholder

    if not predictions:
        placeholder["Notes"] = "No predictions were generated."
        return placeholder

    import io
    import contextlib

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(str(annotation_path))
        coco_dt = coco_gt.loadRes(predictions)

        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        # Scope evaluation to only the images present in predictions so that
        # recall is computed over our subset (e.g. 100 images), not the full
        # val2017 GT (5000 images). Without this, recall is capped at ~2%
        # and mAP reports ~1% even for a perfect model.
        coco_eval.params.imgIds = sorted(
            set(int(p["image_id"]) for p in predictions)
        )
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

    precision_tensor = coco_eval.eval.get("precision")
    recall_tensor = coco_eval.eval.get("recall")

    precision_score: float | None = None

    if precision_tensor is not None and precision_tensor.size > 0:
        precision_values = precision_tensor[0, :, :, 0, 2]
        precision_values = precision_values[precision_values > -1]

        if precision_values.size > 0:
            precision_score = float(np.mean(precision_values))

    recall_score: float | None = None

    if recall_tensor is not None and recall_tensor.size > 0:
        recall_values = recall_tensor[0, :, 0, 2]
        recall_values = recall_values[recall_values > -1]

        if recall_values.size > 0:
            recall_score = float(np.mean(recall_values))

    return {
        "mAP@0.5": float(coco_eval.stats[1]),
        "mAP@0.5:0.95": float(coco_eval.stats[0]),
        "Precision_Score": precision_score,
        "Recall": recall_score,
        "Notes": None,
    }


def get_sample_outputs(
    outputs: dict[str, list[np.ndarray]],
    index: int,
) -> dict[str, list[np.ndarray]]:
    """
    Extract inference outputs corresponding to a single sample index.

    The function creates a new output bundle containing only the arrays
    associated with the specified sample index for each output name.

    Args:
        outputs (dict[str, list[np.ndarray]]): Dictionary mapping output names
            to lists of NumPy arrays.
        index (int): Sample index to extract.

    Returns:
        dict[str, list[np.ndarray]]:
            Dictionary containing the extracted sample outputs.
    """
    sample_bundle: dict[str, list[np.ndarray]] = {}

    for output_name, arrays in outputs.items():
        if index < len(arrays):
            sample_bundle[output_name] = [
                np.asarray(arrays[index])
            ]

    return sample_bundle

In [15]:
metadata_path = VAL_DIR / "metadata.csv"
metadata_df = pd.read_csv(metadata_path) if metadata_path.exists() else pd.DataFrame()

accuracy_records: list[dict[str, Any]] = []
spec_lookup = {spec["Model"]: spec for spec in inference_candidates}
for (model_name, backend), inference_record in inference_results.items():
    spec = spec_lookup.get(model_name, {})
    outputs = inference_record.get("Outputs", {})
    predictions: list[dict[str, Any]] = []

    if outputs and not metadata_df.empty:
        num_samples = min(len(metadata_df), max((len(v) for v in outputs.values()), default=0))
        for sample_index in range(num_samples):
            sample_outputs = get_sample_outputs(outputs, sample_index)
            if not sample_outputs:
                continue
            try:
                predictions.extend(
                    postprocess_outputs(sample_outputs, metadata_df.iloc[sample_index].to_dict())
                )
            except Exception as exc:
                accuracy_records.append(
                    {
                        "Model": model_name,
                        "Precision": spec.get("Precision"),
                        "Backend": backend,
                        "mAP@0.5": None,
                        "mAP@0.5:0.95": None,
                        "Precision_Score": None,
                        "Recall": None,
                        "Notes": f"Post-processing failed: {exc}",
                    }
                )
                predictions = []
                break

    metrics = evaluate_coco_map(predictions, COCO_ANNOTATION_FILE)
    accuracy_records.append(
        {
            "Model": model_name,
            "Precision": spec.get("Precision"),
            "Backend": backend,
            "mAP@0.5": metrics["mAP@0.5"],
            "mAP@0.5:0.95": metrics["mAP@0.5:0.95"],
            "Precision_Score": metrics["Precision_Score"],
            "Recall": metrics["Recall"],
            "Notes": metrics["Notes"] or inference_record.get("Notes"),
        }
    )

accuracy_df = pd.DataFrame(
    accuracy_records,
    columns=[
        "Model",
        "Precision",
        "Backend",
        "mAP@0.5",
        "mAP@0.5:0.95",
        "Precision_Score",
        "Recall",
        "Notes",
    ],
).drop_duplicates(subset=["Model", "Precision", "Backend"], keep="last")

accuracy_df

,Model,Precision,Backend,mAP@0.5,mAP@0.5:0.95,Precision_Score,Recall,Notes
0,LibreYOLOXs QAI Hub FP32 DLC,FP32,CPU,0.383722,0.254136,0.383722,0.430366,None
1,LibreYOLOXs QAI Hub FP32 DLC,FP32,GPU,0.549758,0.403405,0.549758,0.607271,None
2,LibreYOLOXs QAI Hub INT8 DLC,INT8,CPU,0.560268,0.387069,0.560268,0.604831,None
3,LibreYOLOXs QAI Hub INT8 DLC,INT8,HTP/DSP,0.548002,0.388590,0.548002,0.588716,None
4,LibreYOLOXs QAIRT FP32 DLC,FP32,CPU,0.383722,0.254136,0.383722,0.430366,None
5,LibreYOLOXs QAIRT FP32 DLC,FP32,GPU,0.549758,0.403405,0.549758,0.607271,None
6,LibreYOLOXs QAIRT INT8 DLC,INT8,CPU,0.550367,0.371485,0.550367,0.586153,None
7,LibreYOLOXs QAIRT INT8 DLC,INT8,HTP/DSP,0.541550,0.367365,0.541550,0.574555,None
8,LibreYOLOXs SNPE FP32 DLC,FP32,CPU,0.383722,0.254136,0.383722,0.430366,None
9,LibreYOLOXs SNPE FP32 DLC,FP32,GPU,0.549758,0.403405,0.549758,0.607271,None


## Derived Metrics

We now compute the headline quantities that usually drive deployment choices:

- compression ratio
- latency speedup
- throughput gain
- mAP drop
- precision / recall change
- output drift

The helper functions use safe division so the notebook still runs when a metric is missing.

In [16]:
def safe_divide(
    a: Any,
    b: Any,
) -> float | None:
    """
    Safely divide two numeric values.

    The function returns None when:
    - Either value is None
    - Either value is NaN
    - The denominator is zero

    Otherwise, both values are converted to float and divided.

    Args:
        a (Any): Numerator value.
        b (Any): Denominator value.

    Returns:
        float | None:
            - Division result as a float.
            - None if the operation is invalid or unsafe.
    """
    if a is None or b is None:
        return None

    if pd.isna(a) or pd.isna(b) or b == 0:
        return None

    return float(a) / float(b)


def lookup_inventory_size(
    pipeline: str,
    precision: str,
    fmt: str = "DLC",
) -> float | None:
    """
    Retrieve the model file size from the model inventory table.

    The function searches the global `model_inventory_df` DataFrame for a model
    matching the specified pipeline, precision, and format.

    Args:
        pipeline (str): Model pipeline identifier.
        precision (str): Model precision type (e.g., "FP32", "INT8").
        fmt (str, optional): Model format to search for. Defaults to "DLC".

    Returns:
        float | None:
            - File size in megabytes.
            - None if no matching entry is found or the size is missing.
    """
    match = model_inventory_df[
        (model_inventory_df["Pipeline"] == pipeline)
        & (model_inventory_df["Precision"] == precision)
        & (model_inventory_df["Format"] == fmt)
    ]

    if match.empty:
        return None

    value = match.iloc[0]["File_Size_MB"]

    return None if pd.isna(value) else float(value)


def lookup_profile_metric(
    model: str,
    backend: str,
    column: str,
) -> float | None:
    """
    Retrieve a profiling metric for a specific model and backend.

    The function searches the global `profile_results_df` DataFrame for a row
    matching the specified model name and backend, then returns the requested
    metric column as a float.

    Backend matching is case-insensitive.

    Args:
        model (str): Model name to search for.
        backend (str): Backend or runtime name.
        column (str): Metric column name to retrieve.

    Returns:
        float | None:
            - Metric value as a float.
            - None if no matching entry is found or the metric value is missing.
    """
    match = profile_results_df[
        (profile_results_df["Model"] == model)
        & (
            profile_results_df["Backend"].str.upper()
            == backend.upper()
        )
    ]

    if match.empty:
        return None

    value = match.iloc[0][column]

    return None if pd.isna(value) else float(value)


def lookup_accuracy_metric(
    model: str,
    column: str,
) -> float | None:
    """
    Retrieve an accuracy metric for a specific model.

    The function searches the global `accuracy_df` DataFrame for a row matching
    the specified model name and returns the requested metric column as a float.

    Args:
        model (str): Model name to search for.
        column (str): Accuracy metric column name to retrieve.

    Returns:
        float | None:
            - Metric value as a float.
            - None if no matching entry is found or the metric value is missing.
    """
    match = accuracy_df[
        accuracy_df["Model"] == model
    ]

    if match.empty:
        return None

    value = match.iloc[0][column]

    return None if pd.isna(value) else float(value)


def lookup_similarity_metric(
    pipeline: str,
    column: str,
) -> float | None:
    """
    Retrieve the mean similarity metric for a specific pipeline.

    The function searches the global `output_similarity_df` DataFrame for rows
    matching the specified pipeline and computes the mean value of the requested
    metric column.

    Args:
        pipeline (str): Pipeline identifier to search for.
        column (str): Similarity metric column name.

    Returns:
        float | None:
            - Mean metric value as a float.
            - None if no matching entries are found or the metric value is
              missing.
    """
    match = output_similarity_df[
        output_similarity_df["Pipeline"] == pipeline
    ]

    if match.empty:
        return None

    value = match[column].mean()

    return None if pd.isna(value) else float(value)


def best_value_with_label(
    values: dict[str, float | None],
    mode: str = "min",
) -> str:
    """
    Select the best numeric value from a dictionary and return it with its label.

    The function filters out entries with `None` values and selects either the
    minimum or maximum value depending on the specified mode.

    Args:
        values (dict[str, float | None]): Dictionary mapping labels to numeric
            values.
        mode (str, optional): Selection mode:
            - `"min"` selects the smallest value.
            - Any other value selects the largest value.
            Defaults to `"min"`.

    Returns:
        str:
            - Formatted string in the form `"label: value"`.
            - `"N/A"` if no valid numeric values are available.
    """
    valid_items = {
        key: value
        for key, value in values.items()
        if value is not None
    }

    if not valid_items:
        return "N/A"

    if mode == "min":
        key = min(valid_items, key=valid_items.get)
    else:
        key = max(valid_items, key=valid_items.get)

    return f"{key}: {valid_items[key]:.4f}"


def compute_summary_rows() -> list[dict[str, Any]]:
    """
    Compute summary rows highlighting the best model deployment results.

    The function aggregates profiling, inventory, accuracy, compression,
    speedup, quantization, and output similarity metrics across all configured
    model pipelines. It selects the best candidate for each summary metric and
    returns the results as a list of dictionaries suitable for conversion into a
    DataFrame or report table.

    Returns:
        list[dict[str, Any]]: Summary records containing:
            - "Metric" (str): Name of the summary metric.
            - "Value" (float | None): Selected metric value.
            - "Pipeline" (str | None): Pipeline associated with the selected
              value.
            - "Backend" (str | None): Backend, comparison label, or artifact
              type associated with the selected value.
            - "Interpretation" (str): Human-readable explanation of the metric.
    """
    rows: list[dict[str, Any]] = []

    latency_candidates: list[tuple[str, str, float]] = []
    throughput_candidates: list[tuple[str, str, float]] = []
    smallest_candidates: list[tuple[str, str, float]] = []
    accuracy_candidates: list[tuple[str, str, float]] = []
    tradeoff_candidates: list[tuple[str, str, float]] = []
    compression_candidates: list[tuple[str, str, float]] = []
    speedup_cpu_candidates: list[tuple[str, str, float]] = []
    speedup_gpu_candidates: list[tuple[str, str, float]] = []
    map_drop_candidates: list[tuple[str, str, float]] = []
    output_drift_candidates: list[tuple[str, str, float]] = []

    for pipeline in PIPELINES:
        fp32_model = PIPELINE_MODEL_NAMES[pipeline]["fp32"]
        int8_model = PIPELINE_MODEL_NAMES[pipeline]["int8"]

        for backend in ["CPU", "HTP/DSP"]:
            latency = lookup_profile_metric(
                int8_model,
                backend,
                "Latency_ms",
            )
            throughput = lookup_profile_metric(
                int8_model,
                backend,
                "Throughput_img_s",
            )

            if latency is not None:
                latency_candidates.append(
                    (pipeline, backend, latency)
                )

            if throughput is not None:
                throughput_candidates.append(
                    (pipeline, backend, throughput)
                )

        int8_size = lookup_inventory_size(pipeline, "INT8")

        if int8_size is not None:
            smallest_candidates.append(
                (pipeline, "INT8 DLC", int8_size)
            )

        int8_map = lookup_accuracy_metric(
            int8_model,
            "mAP@0.5:0.95",
        )

        if int8_map is not None:
            accuracy_candidates.append(
                (pipeline, "INT8 DLC", int8_map)
            )

        best_latency = min(
            [
                value
                for value in [
                    lookup_profile_metric(
                        int8_model,
                        "CPU",
                        "Latency_ms",
                    ),
                    lookup_profile_metric(
                        int8_model,
                        "HTP/DSP",
                        "Latency_ms",
                    ),
                ]
                if value is not None
            ],
            default=None,
        )

        if int8_map is not None and best_latency not in (None, 0):
            tradeoff_candidates.append(
                (
                    pipeline,
                    "INT8 DLC",
                    int8_map / best_latency,
                )
            )

        fp32_size = lookup_inventory_size(pipeline, "FP32")

        if fp32_size is not None and int8_size not in (None, 0):
            compression_candidates.append(
                (
                    pipeline,
                    "FP32/INT8",
                    fp32_size / int8_size,
                )
            )

        fp32_cpu_latency = lookup_profile_metric(
            fp32_model,
            "CPU",
            "Latency_ms",
        )
        fp32_gpu_latency = lookup_profile_metric(
            fp32_model,
            "GPU",
            "Latency_ms",
        )
        int8_htp_latency = lookup_profile_metric(
            int8_model,
            "HTP/DSP",
            "Latency_ms",
        )

        if (
            fp32_cpu_latency not in (None, 0)
            and int8_htp_latency not in (None, 0)
        ):
            speedup_cpu_candidates.append(
                (
                    pipeline,
                    "HTP/DSP vs FP32 CPU",
                    fp32_cpu_latency / int8_htp_latency,
                )
            )

        if (
            fp32_gpu_latency not in (None, 0)
            and int8_htp_latency not in (None, 0)
        ):
            speedup_gpu_candidates.append(
                (
                    pipeline,
                    "HTP/DSP vs FP32 GPU",
                    fp32_gpu_latency / int8_htp_latency,
                )
            )

        fp32_map = lookup_accuracy_metric(
            fp32_model,
            "mAP@0.5:0.95",
        )

        if fp32_map not in (None, 0) and int8_map is not None:
            map_drop_candidates.append(
                (
                    pipeline,
                    "ΔmAP@0.5:0.95",
                    ((int8_map - fp32_map) / fp32_map) * 100,
                )
            )

        mae = lookup_similarity_metric(pipeline, "MAE")

        if mae is not None:
            output_drift_candidates.append(
                (pipeline, "MAE", mae)
            )

    def choose(
        candidates: list[tuple[str, str, float]],
        mode: str,
    ) -> tuple[str | None, str | None, float | None]:
        """
        Select the best candidate according to the requested optimization mode.

        Args:
            candidates (list[tuple[str, str, float]]): Candidate records in the
                form `(pipeline, label, value)`.
            mode (str): Selection mode. Use `"min"` for the smallest value and
                any other value for the largest value.

        Returns:
            tuple[str | None, str | None, float | None]: Selected candidate, or
            `(None, None, None)` if no candidates are available.
        """
        if not candidates:
            return None, None, None

        index = 2

        winner = (
            min(candidates, key=lambda item: item[index])
            if mode == "min"
            else max(candidates, key=lambda item: item[index])
        )

        return winner

    summary_specs = [
        ("Best Latency", choose(latency_candidates, "min"), "Lower is better."),
        ("Best Throughput", choose(throughput_candidates, "max"), "Higher is better."),
        ("Smallest Model", choose(smallest_candidates, "min"), "Smaller models are easier to deploy."),
        ("Best Accuracy", choose(accuracy_candidates, "max"), "Higher COCO accuracy is better."),
        ("Best Accuracy-Speed Trade-off", choose(tradeoff_candidates, "max"), "Heuristic: mAP@0.5:0.95 / latency."),
        ("Compression Ratio", choose(compression_candidates, "max"), "Higher means stronger size reduction."),
        ("INT8 HTP/DSP Speedup over FP32 CPU", choose(speedup_cpu_candidates, "max"), "Higher means stronger HTP/DSP acceleration."),
        ("INT8 HTP/DSP Speedup over FP32 GPU", choose(speedup_gpu_candidates, "max"), "Higher means stronger HTP/DSP acceleration over FP32 GPU."),
        ("mAP Drop after Quantization", choose(map_drop_candidates, "max"), "Negative values mean accuracy decreased."),
        ("Output Drift", choose(output_drift_candidates, "min"), "Lower is better."),
    ]

    for metric, winner, interpretation in summary_specs:
        pipeline, backend_or_label, value = winner

        rows.append(
            {
                "Metric": metric,
                "Value": value,
                "Pipeline": pipeline,
                "Backend": backend_or_label,
                "Interpretation": interpretation,
            }
        )

    return rows

In [17]:
PIPELINES = ["QAI Hub", "QAIRT", "SNPE"]

PIPELINE_MODEL_NAMES = {
    "QAI Hub": {
        "fp32": "LibreYOLOXs QAI Hub FP32 DLC",
        "int8": "LibreYOLOXs QAI Hub INT8 DLC",
    },
    "QAIRT": {
        "fp32": "LibreYOLOXs QAIRT FP32 DLC",
        "int8": "LibreYOLOXs QAIRT INT8 DLC",
    },
    "SNPE": {
        "fp32": "LibreYOLOXs SNPE FP32 DLC",
        "int8": "LibreYOLOXs SNPE INT8 DLC",
    },
}

comparison_records: list[dict[str, Any]] = []
for pipeline in PIPELINES:
    fp32_model = PIPELINE_MODEL_NAMES[pipeline]["fp32"]
    int8_model = PIPELINE_MODEL_NAMES[pipeline]["int8"]

    fp32_dlc_cpu_latency = lookup_profile_metric(fp32_model, "CPU", "Latency_ms")
    fp32_dlc_gpu_latency = lookup_profile_metric(fp32_model, "GPU", "Latency_ms")
    int8_dlc_cpu_latency = lookup_profile_metric(int8_model, "CPU", "Latency_ms")
    int8_dlc_htp_latency = lookup_profile_metric(int8_model, "HTP/DSP", "Latency_ms")

    fp32_dlc_cpu_throughput = lookup_profile_metric(fp32_model, "CPU", "Throughput_img_s")
    fp32_dlc_gpu_throughput = lookup_profile_metric(fp32_model, "GPU", "Throughput_img_s")
    int8_dlc_cpu_throughput = lookup_profile_metric(int8_model, "CPU", "Throughput_img_s")
    int8_dlc_htp_throughput = lookup_profile_metric(int8_model, "HTP/DSP", "Throughput_img_s")

    fp32_map50 = lookup_accuracy_metric(fp32_model, "mAP@0.5")
    fp32_map5095 = lookup_accuracy_metric(fp32_model, "mAP@0.5:0.95")
    fp32_precision = lookup_accuracy_metric(fp32_model, "Precision_Score")
    fp32_recall = lookup_accuracy_metric(fp32_model, "Recall")

    int8_map50 = lookup_accuracy_metric(int8_model, "mAP@0.5")
    int8_map5095 = lookup_accuracy_metric(int8_model, "mAP@0.5:0.95")
    int8_precision = lookup_accuracy_metric(int8_model, "Precision_Score")
    int8_recall = lookup_accuracy_metric(int8_model, "Recall")

    fp32_size = lookup_inventory_size(pipeline, "FP32")
    int8_size = lookup_inventory_size(pipeline, "INT8")

    compression_ratio = safe_divide(fp32_size, int8_size)
    latency_speedup_cpu = safe_divide(fp32_dlc_cpu_latency, int8_dlc_cpu_latency)
    latency_speedup_htp = safe_divide(fp32_dlc_cpu_latency, int8_dlc_htp_latency)
    throughput_gain_cpu = safe_divide(int8_dlc_cpu_throughput, fp32_dlc_cpu_throughput)
    throughput_gain_htp = safe_divide(int8_dlc_htp_throughput, fp32_dlc_cpu_throughput)
    map50_change_pct = None if safe_divide(fp32_map50, 1) is None else ((int8_map50 - fp32_map50) / fp32_map50) * 100 if fp32_map50 not in (None, 0) else None
    map5095_change_pct = None if safe_divide(fp32_map5095, 1) is None else ((int8_map5095 - fp32_map5095) / fp32_map5095) * 100 if fp32_map5095 not in (None, 0) else None
    precision_change_pct = None if safe_divide(fp32_precision, 1) is None else ((int8_precision - fp32_precision) / fp32_precision) * 100 if fp32_precision not in (None, 0) else None
    recall_change_pct = None if safe_divide(fp32_recall, 1) is None else ((int8_recall - fp32_recall) / fp32_recall) * 100 if fp32_recall not in (None, 0) else None
    mae_drift = lookup_similarity_metric(pipeline, "MAE")
    rmse_drift = lookup_similarity_metric(pipeline, "RMSE")
    max_drift = lookup_similarity_metric(pipeline, "Max_Abs_Error")
    cosine_similarity = lookup_similarity_metric(pipeline, "Cosine_Similarity")

    metric_rows = [
        (
            "Model Size",
            "MB",
            [fp32_size, fp32_size, int8_size, int8_size],
            "Smaller values are better for edge deployment.",
            best_value_with_label(
                {
                    f"{pipeline} FP32 DLC": fp32_size,
                    f"{pipeline} INT8 DLC": int8_size,
                },
                mode="min",
            ),
        ),
        (
            "Latency",
            "ms",
            [
                fp32_dlc_cpu_latency,
                fp32_dlc_gpu_latency,
                int8_dlc_cpu_latency,
                int8_dlc_htp_latency,
            ],
            "Lower latency means faster per-image execution.",
            best_value_with_label(
                {
                    "FP32 DLC CPU": fp32_dlc_cpu_latency,
                    "FP32 DLC GPU": fp32_dlc_gpu_latency,
                    "INT8 DLC CPU": int8_dlc_cpu_latency,
                    "INT8 DLC HTP/DSP": int8_dlc_htp_latency,
                },
                mode="min",
            ),
        ),
        (
            "Throughput",
            "img/s",
            [
                fp32_dlc_cpu_throughput,
                fp32_dlc_gpu_throughput,
                int8_dlc_cpu_throughput,
                int8_dlc_htp_throughput,
            ],
            "Higher throughput is better for sustained workloads.",
            best_value_with_label(
                {
                    "FP32 DLC CPU": fp32_dlc_cpu_throughput,
                    "FP32 DLC GPU": fp32_dlc_gpu_throughput,
                    "INT8 DLC CPU": int8_dlc_cpu_throughput,
                    "INT8 DLC HTP/DSP": int8_dlc_htp_throughput,
                },
                mode="max",
            ),
        ),
        (
            "mAP@0.5",
            "score",
            [fp32_map50, fp32_map50, int8_map50, int8_map50],
            "Higher values indicate better object detection quality at IoU 0.5.",
            best_value_with_label(
                {
                    "FP32 DLC": fp32_map50,
                    "INT8 DLC": int8_map50,
                },
                mode="max",
            ),
        ),
        (
            "mAP@0.5:0.95",
            "score",
            [fp32_map5095, fp32_map5095, int8_map5095, int8_map5095],
            "This is the strictest COCO summary metric.",
            best_value_with_label(
                {
                    "FP32 DLC": fp32_map5095,
                    "INT8 DLC": int8_map5095,
                },
                mode="max",
            ),
        ),
        (
            "Precision",
            "score",
            [fp32_precision, fp32_precision, int8_precision, int8_precision],
            "Precision summarizes how often predicted detections are correct.",
            best_value_with_label(
                {
                    "FP32 DLC": fp32_precision,
                    "INT8 DLC": int8_precision,
                },
                mode="max",
            ),
        ),
        (
            "Recall",
            "score",
            [fp32_recall, fp32_recall, int8_recall, int8_recall],
            "Recall summarizes how many ground-truth objects were recovered.",
            best_value_with_label(
                {
                    "FP32 DLC": fp32_recall,
                    "INT8 DLC": int8_recall,
                },
                mode="max",
            ),
        ),
        (
            "Mean Absolute Output Difference",
            "abs",
            [None, None, mae_drift, mae_drift],
            "Lower output drift indicates closer numerical agreement after quantization.",
            best_value_with_label({f"{pipeline} INT8 vs FP32": mae_drift}, mode="min"),
        ),
        (
            "RMSE Output Difference",
            "abs",
            [None, None, rmse_drift, rmse_drift],
            "RMSE emphasizes larger output deviations.",
            best_value_with_label({f"{pipeline} INT8 vs FP32": rmse_drift}, mode="min"),
        ),
        (
            "Max Output Difference",
            "abs",
            [None, None, max_drift, max_drift],
            "Tracks the worst observed numerical drift.",
            best_value_with_label({f"{pipeline} INT8 vs FP32": max_drift}, mode="min"),
        ),
        (
            "Cosine Similarity",
            "score",
            [None, None, cosine_similarity, cosine_similarity],
            "Values closer to 1 mean the output direction is preserved.",
            best_value_with_label({f"{pipeline} INT8 vs FP32": cosine_similarity}, mode="max"),
        ),
        (
            "Compression Ratio",
            "x",
            [None, None, compression_ratio, compression_ratio],
            "Higher compression means the INT8 artifact is smaller relative to FP32.",
            best_value_with_label({f"{pipeline} FP32/INT8": compression_ratio}, mode="max"),
        ),
        (
            "Latency Speedup",
            "x",
            [None, None, latency_speedup_cpu, latency_speedup_htp],
            "Values above 1 mean INT8 is faster than the chosen FP32 reference.",
            best_value_with_label(
                {
                    "INT8 CPU vs FP32 CPU": latency_speedup_cpu,
                    "INT8 HTP/DSP vs FP32 CPU": latency_speedup_htp,
                },
                mode="max",
            ),
        ),
        (
            "Throughput Gain",
            "x",
            [None, None, throughput_gain_cpu, throughput_gain_htp],
            "Values above 1 mean INT8 increased processed images per second.",
            best_value_with_label(
                {
                    "INT8 CPU vs FP32 CPU": throughput_gain_cpu,
                    "INT8 HTP/DSP vs FP32 CPU": throughput_gain_htp,
                },
                mode="max",
            ),
        ),
        (
            "Accuracy Drop",
            "%",
            [None, None, map5095_change_pct, map5095_change_pct],
            "Negative values indicate a drop in COCO accuracy after quantization.",
            best_value_with_label({f"{pipeline} ΔmAP@0.5:0.95": map5095_change_pct}, mode="max"),
        ),
    ]

    for metric_name, unit, values, interpretation, best_result in metric_rows:
        comparison_records.append(
            {
                "Pipeline": pipeline,
                "Metric": metric_name,
                "Unit": unit,
                "FP32_DLC_CPU": values[0],
                "FP32_DLC_GPU": values[1],
                "INT8_DLC_CPU": values[2],
                "INT8_DLC_HTP_DSP": values[3],
                "Best_Result": best_result,
                "Interpretation": interpretation,
            }
        )

comparison_df = pd.DataFrame(comparison_records)
summary_df = pd.DataFrame(compute_summary_rows())

## Final Comparison Table

The main benchmark table, `comparison_df`, is organized **per pipeline** so that:

- the DLC columns correspond to the current pipeline (`QAI Hub`, `QAIRT`, or `SNPE`)
- the requested backend-oriented columns remain intact for discussion

In [18]:
comparison_df.fillna("—")

,Pipeline,Metric,Unit,FP32_DLC_CPU,FP32_DLC_GPU,INT8_DLC_CPU,INT8_DLC_HTP_DSP,Best_Result,Interpretation
0,QAI Hub,Model Size,MB,34.509594,34.509594,9.518322,9.518322,QAI Hub INT8 DLC: 9.5183,Smaller values are better for edge deployment.
1,QAI Hub,Latency,ms,635.783,207.754,179.081000,6.817000,INT8 DLC HTP/DSP: 6.8170,Lower latency means faster per-image execution.
2,QAI Hub,Throughput,img/s,1.573,4.813,5.584000,146.692000,INT8 DLC HTP/DSP: 146.6920,Higher throughput is better for sustained work...
3,QAI Hub,mAP@0.5,score,0.383722,0.383722,0.560268,0.560268,INT8 DLC: 0.5603,Higher values indicate better object detection...
4,QAI Hub,mAP@0.5:0.95,score,0.254136,0.254136,0.387069,0.387069,INT8 DLC: 0.3871,This is the strictest COCO summary metric.
5,QAI Hub,Precision,score,0.383722,0.383722,0.560268,0.560268,INT8 DLC: 0.5603,Precision summarizes how often predicted detec...
6,QAI Hub,Recall,score,0.430366,0.430366,0.604831,0.604831,INT8 DLC: 0.6048,Recall summarizes how many ground-truth object...
7,QAI Hub,Mean Absolute Output Difference,abs,—,—,7.349444,7.349444,QAI Hub INT8 vs FP32: 7.3494,Lower output drift indicates closer numerical ...
8,QAI Hub,RMSE Output Difference,abs,—,—,17.241268,17.241268,QAI Hub INT8 vs FP32: 17.2413,RMSE emphasizes larger output deviations.
9,QAI Hub,Max Output Difference,abs,—,—,313.942123,313.942123,QAI Hub INT8 vs FP32: 313.9421,Tracks the worst observed numerical drift.


## Executive Summary Table

In [19]:
summary_df

,Metric,Value,Pipeline,Backend,Interpretation
0,Best Latency,6.691000,SNPE,HTP/DSP,Lower is better.
1,Best Throughput,149.454000,SNPE,HTP/DSP,Higher is better.
2,Smallest Model,8.841442,QAIRT,INT8 DLC,Smaller models are easier to deploy.
3,Best Accuracy,0.387069,QAI Hub,INT8 DLC,Higher COCO accuracy is better.
4,Best Accuracy-Speed Trade-off,0.056780,QAI Hub,INT8 DLC,Heuristic: mAP@0.5:0.95 / latency.
5,Compression Ratio,3.903094,QAIRT,FP32/INT8,Higher means stronger size reduction.
6,INT8 HTP/DSP Speedup over FP32 CPU,95.737707,SNPE,HTP/DSP vs FP32 CPU,Higher means stronger HTP/DSP acceleration.
7,INT8 HTP/DSP Speedup over FP32 GPU,31.741145,SNPE,HTP/DSP vs FP32 GPU,Higher means stronger HTP/DSP acceleration ove...
8,mAP Drop after Quantization,52.307859,QAI Hub,ΔmAP@0.5:0.95,Negative values mean accuracy decreased.
9,Output Drift,7.349444,QAI Hub,MAE,Lower is better.


## Visualization

The analysis closes the loop visually with plots for:

- model size comparison
- latency by backend
- throughput by backend
- mAP comparison
- speedup comparison
- accuracy vs latency scatter
- output similarity bar chart

In [20]:
plt.style.use("default")

def save_current_figure(filename: str) -> Path:
    """
    Save the currently active Matplotlib figure to disk.

    The function applies a tight layout, saves the figure using the configured
    output directory, closes the figure to release resources, and returns the
    generated file path.

    Args:
        filename (str): Output filename for the saved figure.

    Returns:
        Path: Path to the saved figure file.
    """
    figure_path = FIGURES_DIR / filename

    plt.tight_layout()

    plt.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    return figure_path


def plot_model_sizes() -> Path | None:
    """
    Generate and save a bar chart comparing model file sizes.

    The function filters models with valid file size information from the
    global `model_inventory_df` DataFrame and creates a bar chart showing the
    size of each model in megabytes.

    Returns:
        Path | None:
            - Path to the saved figure if data is available.
            - None if no valid model size data exists.
    """
    data = model_inventory_df.dropna(subset=["File_Size_MB"])

    if data.empty:
        return None

    plt.figure(figsize=(12, 5))

    plt.bar(
        data["Model"],
        data["File_Size_MB"],
    )

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Size (MB)")
    plt.title("Model Size Comparison")

    return save_current_figure("model_size_comparison.png")


def plot_profile_metric(
    metric: str,
    title: str,
    filename: str,
    ylabel: str,
) -> Path | None:
    """
    Generate and save a bar chart for a profiling metric grouped by backend.

    The function filters rows with valid metric values from the global
    `profile_results_df` DataFrame, creates a pivot table grouped by model and
    backend, and generates a bar chart visualization.

    Args:
        metric (str): Column name of the profiling metric to visualize.
        title (str): Plot title.
        filename (str): Output filename for the saved figure.
        ylabel (str): Label for the Y-axis.

    Returns:
        Path | None:
            - Path to the saved figure if data is available.
            - None if no valid metric data exists.
    """
    data = profile_results_df.dropna(subset=[metric])

    if data.empty:
        return None

    pivot = data.pivot_table(
        index="Model",
        columns="Backend",
        values=metric,
        aggfunc="mean",
    )

    pivot.plot(
        kind="bar",
        figsize=(12, 6),
    )

    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")

    return save_current_figure(filename)


def plot_map_comparison() -> Path | None:
    """
    Generate and save a bar chart comparing COCO mAP scores across models.

    The function filters rows with valid `mAP@0.5:0.95` values from the global
    `accuracy_df` DataFrame and creates a bar chart showing model accuracy
    performance.

    Returns:
        Path | None:
            - Path to the saved figure if data is available.
            - None if no valid accuracy data exists.
    """
    data = accuracy_df.dropna(subset=["mAP@0.5:0.95"])

    if data.empty:
        return None

    plt.figure(figsize=(10, 5))

    plt.bar(
        data["Model"],
        data["mAP@0.5:0.95"],
    )

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("mAP@0.5:0.95")
    plt.title("mAP Comparison")

    return save_current_figure("map_comparison.png")


def plot_speedup_comparison() -> Path | None:
    """
    Generate and save a bar chart comparing INT8 latency speedups.

    The function extracts latency speedup metrics from the global
    `comparison_df` DataFrame, reshapes the data by backend, and generates a
    grouped bar chart comparing INT8 CPU and HTP/DSP speedups across pipelines.

    Returns:
        Path | None:
            - Path to the saved figure if valid speedup data is available.
            - None if no valid speedup data exists.
    """
    data = comparison_df[
        comparison_df["Metric"] == "Latency Speedup"
    ].copy()

    if data.empty:
        return None

    columns = [
        "INT8_DLC_CPU",
        "INT8_DLC_HTP_DSP",
    ]

    melted = data.melt(
        id_vars=["Pipeline"],
        value_vars=columns,
        var_name="Backend",
        value_name="Speedup",
    )

    melted = melted.dropna(subset=["Speedup"])

    if melted.empty:
        return None

    pivot = melted.pivot_table(
        index="Pipeline",
        columns="Backend",
        values="Speedup",
        aggfunc="mean",
    )

    pivot.plot(
        kind="bar",
        figsize=(10, 5),
    )

    plt.title("INT8 Latency Speedup Comparison")
    plt.ylabel("Speedup (x)")
    plt.xticks(rotation=0)

    return save_current_figure("speedup_comparison.png")


def plot_accuracy_vs_latency() -> Path | None:
    """
    Generate and save a scatter plot comparing accuracy versus latency.

    The function computes the best INT8 latency for each pipeline across CPU,
    GPU, and HTP/DSP backends, pairs it with the corresponding
    `mAP@0.5:0.95` accuracy metric, and visualizes the trade-off between
    latency and detection performance.

    Returns:
        Path | None:
            - Path to the saved figure if valid data is available.
            - None if no valid accuracy-latency pairs exist.
    """
    rows: list[dict[str, float | str]] = []

    for pipeline in PIPELINES:
        int8_model = PIPELINE_MODEL_NAMES[pipeline]["int8"]

        accuracy = lookup_accuracy_metric(
            int8_model,
            "mAP@0.5:0.95",
        )

        best_latency = min(
            [
                value
                for value in [
                    lookup_profile_metric(
                        int8_model,
                        "CPU",
                        "Latency_ms",
                    ),
                    lookup_profile_metric(
                        int8_model,
                        "GPU",
                        "Latency_ms",
                    ),
                    lookup_profile_metric(
                        int8_model,
                        "HTP/DSP",
                        "Latency_ms",
                    ),
                ]
                if value is not None
            ],
            default=None,
        )

        if accuracy is not None and best_latency is not None:
            rows.append(
                {
                    "Pipeline": pipeline,
                    "Latency_ms": best_latency,
                    "mAP@0.5:0.95": accuracy,
                }
            )

    if not rows:
        return None

    data = pd.DataFrame(rows)

    plt.figure(figsize=(8, 6))

    plt.scatter(
        data["Latency_ms"],
        data["mAP@0.5:0.95"],
    )

    for _, row in data.iterrows():
        plt.annotate(
            row["Pipeline"],
            (
                row["Latency_ms"],
                row["mAP@0.5:0.95"],
            ),
        )

    plt.xlabel("Best INT8 Latency (ms)")
    plt.ylabel("mAP@0.5:0.95")
    plt.title("Accuracy vs Latency")

    return save_current_figure("accuracy_vs_latency.png")


def plot_output_similarity() -> Path | None:
    """
    Generate and save a bar chart summarizing output similarity metrics.

    The function aggregates output similarity metrics by pipeline using the
    global `output_similarity_df` DataFrame and visualizes the mean values of:
    - MAE
    - RMSE
    - Cosine Similarity

    Returns:
        Path | None:
            - Path to the saved figure if valid similarity data is available.
            - None if no similarity data exists.
    """
    if output_similarity_df.empty:
        return None

    metrics = (
        output_similarity_df.groupby("Pipeline")[
            ["MAE", "RMSE", "Cosine_Similarity"]
        ]
        .mean(numeric_only=True)
    )

    if metrics.empty:
        return None

    metrics.plot(
        kind="bar",
        figsize=(10, 5),
    )

    plt.title("Output Similarity Summary")
    plt.ylabel("Value")
    plt.xticks(rotation=0)

    return save_current_figure("output_similarity.png")

In [21]:
generated_figures = {
    "model_size": plot_model_sizes(),
    "latency": plot_profile_metric("Latency_ms", "Latency by Backend", "latency_by_backend.png", "Latency (ms)"),
    "throughput": plot_profile_metric("Throughput_img_s", "Throughput by Backend", "throughput_by_backend.png", "Images / second"),
    "map": plot_map_comparison(),
    "speedup": plot_speedup_comparison(),
    "accuracy_latency": plot_accuracy_vs_latency(),
    "output_similarity": plot_output_similarity(),
}

pd.DataFrame(
    [{"Figure": key, "Path": str(path) if path is not None else None} for key, path in generated_figures.items()]
)

,Figure,Path
0,model_size,reports/figures/model_size_comparison.png
1,latency,reports/figures/latency_by_backend.png
2,throughput,reports/figures/throughput_by_backend.png
3,map,reports/figures/map_comparison.png
4,speedup,reports/figures/speedup_comparison.png
5,accuracy_latency,reports/figures/accuracy_vs_latency.png
6,output_similarity,reports/figures/output_similarity.png


## Save All Results

All tabular outputs are saved to `reports/tables/`, while the summary is exported to `reports/summary.md`.

In [22]:
def format_value(
    value: Any,
    unit: str | None = None,
) -> str:
    """
    Format a value as a human-readable string with optional units.

    The function:
    - Returns `"N/A"` for None or NaN values.
    - Formats floating-point values with four decimal places.
    - Converts all other values to strings.
    - Optionally appends a unit suffix.

    Args:
        value (Any): Value to format.
        unit (str | None, optional): Unit suffix to append to the formatted
            value. Defaults to None.

    Returns:
        str: Formatted value string.
    """
    if value is None or (
        isinstance(value, float)
        and math.isnan(value)
    ):
        return "N/A"

    if isinstance(value, float):
        text = f"{value:.4f}"
    else:
        text = str(value)

    return f"{text} {unit}".strip() if unit else text

In [23]:
profile_results_df.to_csv(TABLES_DIR / "profile_results_df.csv", index=False)
output_similarity_df.to_csv(TABLES_DIR / "output_similarity_df.csv", index=False)
accuracy_df.to_csv(TABLES_DIR / "accuracy_df.csv", index=False)
comparison_df.to_csv(TABLES_DIR / "comparison_df.csv", index=False)
summary_df.to_csv(TABLES_DIR / "summary_df.csv", index=False)

summary_lines = [
    "# LibreYOLOXs Benchmark Summary",
    "",
    "## Optional dependency status",
    OPTIONAL_DEPENDENCIES_DF.to_markdown(index=False),
    "",
    "## Model inventory",
    model_inventory_df.to_markdown(index=False),
    "",
    "## Profiling results",
    profile_results_df.to_markdown(index=False),
    "",
    "## Accuracy results",
    accuracy_df.to_markdown(index=False),
    "",
    "## Summary table",
    summary_df.to_markdown(index=False),
]

(REPORTS_ROOT / "summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

print(f"Saved tables to: {TABLES_DIR}")
print(f"Saved markdown summary to: {REPORTS_ROOT / 'summary.md'}")

Saved tables to: reports/tables
Saved markdown summary to: reports/summary.md


## Final Markdown Interpretation

The final cell synthesizes the benchmark. If a metric is unavailable, the text falls back to placeholders instead of failing.

In [24]:
def lookup_summary(metric_name: str) -> dict[str, Any]:
    """
    Retrieve a summary metric record from the summary table.

    The function searches the global `summary_df` DataFrame for a row matching
    the specified metric name and returns the row as a dictionary.

    Args:
        metric_name (str): Name of the summary metric to retrieve.

    Returns:
        dict[str, Any]:
            - Dictionary containing the summary row values.
            - If the metric is not found, returns a placeholder dictionary with
              `None` values for `"Value"`, `"Pipeline"`, and `"Backend"`.
    """
    match = summary_df[
        summary_df["Metric"] == metric_name
    ]

    if match.empty:
        return {
            "Value": None,
            "Pipeline": None,
            "Backend": None,
        }

    row = match.iloc[0]

    return row.to_dict()

In [25]:
best_latency = lookup_summary("Best Latency")
best_throughput = lookup_summary("Best Throughput")
smallest_model = lookup_summary("Smallest Model")
best_accuracy = lookup_summary("Best Accuracy")
best_tradeoff = lookup_summary("Best Accuracy-Speed Trade-off")
compression_ratio = lookup_summary("Compression Ratio")
htp_speedup_cpu = lookup_summary("INT8 HTP/DSP Speedup over FP32 CPU")
htp_speedup_gpu = lookup_summary("INT8 HTP/DSP Speedup over FP32 GPU")
map_drop = lookup_summary("mAP Drop after Quantization")
output_drift = lookup_summary("Output Drift")

interpretation = f"""
### Final Interpretation

- Quantization reduced model size by **{format_value(compression_ratio.get('Value'), 'x')}** in the best observed pipeline (**{compression_ratio.get('Pipeline') or 'N/A'}**).
- INT8 improved latency to a best observed value of **{format_value(best_latency.get('Value'), 'ms')}** on **{best_latency.get('Pipeline') or 'N/A'} / {best_latency.get('Backend') or 'N/A'}**.
- INT8 improved throughput to **{format_value(best_throughput.get('Value'), 'img/s')}** on **{best_throughput.get('Pipeline') or 'N/A'} / {best_throughput.get('Backend') or 'N/A'}**.
- The best observed COCO accuracy was **{format_value(best_accuracy.get('Value'))}** on **{best_accuracy.get('Pipeline') or 'N/A'}**.
- The best accuracy-speed trade-off was **{format_value(best_tradeoff.get('Value'))}** on **{best_tradeoff.get('Pipeline') or 'N/A'}**.
- INT8 HTP/DSP speedup over FP32 CPU reached **{format_value(htp_speedup_cpu.get('Value'), 'x')}**.
- INT8 HTP/DSP speedup over FP32 GPU reached **{format_value(htp_speedup_gpu.get('Value'), 'x')}**.
- mAP changed by **{format_value(map_drop.get('Value'), '%')}** after quantization in the strongest measured pipeline.
- Output drift (measured here as MAE) was **{format_value(output_drift.get('Value'))}**.
- The current best deployment candidate is **{best_tradeoff.get('Pipeline') or 'N/A'}**.
- The recommended deployment backend for Qualcomm edge devices is **{best_latency.get('Backend') or 'N/A'}**, subject to application-specific accuracy constraints.

**Note:** if any metric is missing, inspect the saved reports under `reports/` to determine whether the gap came from unavailable data, unsupported QAI Hub runtime combinations, missing local CLIs, or model-specific output decoding that needs adaptation.
"""

if display is not None and Markdown is not None:
    display(Markdown(interpretation))
else:
    print(interpretation)


### Final Interpretation

- Quantization reduced model size by **3.9031 x** in the best observed pipeline (**QAIRT**).
- INT8 improved latency to a best observed value of **6.6910 ms** on **SNPE / HTP/DSP**.
- INT8 improved throughput to **149.4540 img/s** on **SNPE / HTP/DSP**.
- The best observed COCO accuracy was **0.3871** on **QAI Hub**.
- The best accuracy-speed trade-off was **0.0568** on **QAI Hub**.
- INT8 HTP/DSP speedup over FP32 CPU reached **95.7377 x**.
- INT8 HTP/DSP speedup over FP32 GPU reached **31.7411 x**.
- mAP changed by **52.3079 %** after quantization in the strongest measured pipeline.
- Output drift (measured here as MAE) was **7.3494**.
- The current best deployment candidate is **QAI Hub**.
- The recommended deployment backend for Qualcomm edge devices is **HTP/DSP**, subject to application-specific accuracy constraints.

**Note:** if any metric is missing, inspect the saved reports under `reports/` to determine whether the gap came from unavailable data, unsupported QAI Hub runtime combinations, missing local CLIs, or model-specific output decoding that needs adaptation.
